<font size="8">**Modulo 1.1 — Dallo *script* allo *strumento***</font><br>

> Corso Python per sistemisti · Giorno 1 · monitoraggio, affidabilità e troubleshooting


> (c) 2026 Antonio Michele Piemontese




Finora avete usato Python per **scripting**: probabilmente programmi brevi, lineari, scritti in fretta per
risolvere un compito puntuale. Funzionano — ma uno script di monitoraggio che gira in
produzione viene **rieseguito** ogni notte, **modificato** a mesi di distanza, **condiviso**
nel team e applicato a **centinaia di macchine**. In quel contesto "funziona sul mio caso"
non basta più: serve uno *strumento* affidabile, leggibile e riproducibile.

Questo modulo prende uno script realistico (speriamo simile al vostro!) e lo trasforma. Il filo conduttore — che giustifica
ogni passaggio — è **uno solo**, e lo ritroverete in tutto il corso:

1. **Separare la logica dall'I/O.** Il *calcolo* sta in funzioni **pure** (stesso input → stesso
   output, nessun effetto collaterale); leggere file, interrogare DB e agire sui sistemi sta
   ai **bordi**. La logica isolata è leggibile e, soprattutto, **testabile**.
2. **Gestire gli errori in modo esplicito.** Tutto ciò che tocca file, rete, DB o sistemi
   *fallisce*: una riga malformata, un host irraggiungibile, un timeout. Lo strumento deve
   **sopravvivere** e dirti *cosa* è andato storto, non interrompersi in silenzio.
3. **Rendere tutto riproducibile.** Niente percorsi e soglie cablati nel codice: parametri,
   configurazione, e un punto di ingresso pulito che si integra con cron/scheduler.

Costruiamo lo strumento di monitoraggio infrastruttura v1 (nei prossimi giorni vedremo la v2 e v3).

Seguiamo lo schema **as-is (stile scripting) → strumento (stile idiomatico)**: prima la versione com'è probabilmente scritta oggi (dai sistemisti), poi la stessa
attività ristrutturata, un *concern* alla volta.

# Rilevamento ambiente di sviluppo

In [1]:
# impostazione del TOGGLE BINARIO:
try:
    import google.colab                      # package disponibile SOLO in Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running on Colab:", IN_COLAB)


# IMPORT dei package necessari (necessari sia in VSC che in Colab perle IMMAGINI):
from IPython.display import Image, display   # import dei package di incorporamento e visualizzazione immagine (una tantum)
                                             # Image e display sono entrambi necessari a Jupyter Notebook
                                             # Google Colab utilizza solo Image
import os                                    # necessario a Google Colab per vedere da una cella codice
                                             # i contenuti del 'content'

Running on Colab: True


# Immagini

Il notebook incorpora le seguenti immagini *png* (in ordine di incorporamento):

* `mappa_idioma.png`



# Legenda delle icone (standard)
Legenda delle icone (standard) usate nel notebook (per garantire consistenza semantica):

👉 punto di attenzione, il "succo"<br>
📌 nota / inizio di una nota<br>
📦 punto elenco importante<br>
📊 dati/numeri<br>
🔹 punto elenco normale<br>
⭐ punto elenco importante<br>
✅ punto risolto, positivo<br>
❌ punto negativo, da evitare<br>
⚠️ attenzione, warning, allarme<br>
💡 idea chiave<br>
🧠 idea intuitiva<br>
📝 sintesi / bottom-line<br>
⟶ conseguenza, effetto, passo successivo

# Lo scenario

Avete un gruppo di server applicativi che scrivono un **log di accesso**. Ogni riga è una richiesta, in formato `chiave=valore` (lo stesso stile dei campi di [Splunk](https://www.splunk.com/)):

```
2025-06-20T14:00:03 host=web01 status=200 rt_ms=47
```

- `host` — il server che ha servito la richiesta
- `status` — il codice HTTP (un `5xx` è un errore lato server)
- `rt_ms` — il *response time* in millisecondi

---
> 📌In questo contesto "richiesta" è una **richiesta HTTP**: un client (un browser, un'app, un altro servizio) chiede qualcosa a un server applicativo, e il server risponde. È l'unità elementare del traffico web.
>
>Ogni volta che accade, il server scrive **una riga** nel log di accesso per registrare quella singola interazione. Quindi nel notebook vale l'equazione: *una riga di log = una richiesta servita*. Se `web01` ha 200 righe, ha gestito 200 richieste.
>
> Guardando i campi che usiamo, ogni riga descrive **l'esito** di quella interazione:
>
>- `host=web01` — quale server ha gestito la richiesta
>- `status=200` — com'è andata, col codice di stato HTTP della risposta (200 = OK, 500 = errore del server, ecc.)
>- `rt_ms=47` — quanto è durata, il *response time* in millisecondi (il tempo che il server ha impiegato a rispondere)
>
>
> <u>Un esempio concreto</u>: un utente apre una pagina web. Il browser manda una richiesta `GET /home` a `web01`; il server la elabora in 47 ms e risponde "200 OK". Il server annota l'accaduto come:<br>
> `... host=web01 status=200 rt_ms=47`.<br>
> Quella riga *è* la richiesta, dal punto di vista del log.
>
> Una singola pagina web "pesante" può generare *decine* di richieste (l'HTML, poi ogni immagine, foglio di stile, script, chiamata API...), ciascuna con la sua riga. Per questo i log di accesso crescono in fretta — ed è esattamente da lì che nascono le metriche di affidabilità del modulo: contando le righe ottieniamo il **volume di richieste**, contando quelle con `status` 5xx ottieniamo l'**error rate**, e guardando la distribuzione di `rt_ms` ottieniamo la **latenza** (inclusa la p95). Tre KPI che escono tutti dalla stessa, semplice unità: la richiesta.
>
> Una precisazione: nel notebook le richieste sono **simulate** dalla funzione `genera_righe` per avere dati su cui lavorare, ma un vero server web (nginx, Apache, un application server Java...) produce log dello stesso tipo **in modo nativo**, una riga per ogni richiesta ricevuta.

---

**Obiettivo del tool (lo strumento di monitoraggio):** leggere tutti i log, e per ciascun host calcolare **numero di richieste**,
**error rate** (percentuale di `5xx`) e **latenza p95** (il 95° percentile dei tempi di risposta).
Poi segnalare gli host che superano le soglie — che è il pane quotidiano del monitoraggio di
prestazioni e affidabilità.

> Usiamo il **p95** e non la media perché la media nasconde le code: è il 5% di richieste lente
> a far arrabbiare gli utenti, e la media le annega. Su questo torneremo nel Giorno 2.

## I dati di esempio

Generiamo due cartelle, per raccontare uno scenario che, come sistemisti, conoscete bene:

- **`logs/`** — i log "puliti" su cui lo script gira in **test**: tutto regolare.
- **`logs_reali/`** — gli stessi dati più la **realtà della produzione**: qualche riga
  malformata (campo mancante, valore non numerico, riga spazzatura) e un file estraneo
  finito nella cartella.

`web02` è volutamente il server **sofferente** (più errori, più code di latenza).

---
**📝 Python generale**

Le linee guida PEP sulla scrittura del codice Python - in particolare la [PEP8](https://peps.python.org/pep-0008/) -raccomandano di mettere tutte le import **ad inizio notebook**, e non sparse nel codice quando servono, per avere **una vista d'insieme** dei package utilizzati dal notebook.

---

# Import package

già **pre-installati in Google Colab** oppure appartenenti alla [standard library di Python](https://docs.python.org/3/library/index.html), e che quindi non richiedono `pip install <nome-package>`.

## Esempi di package della python standard library
In merito vedi anche [questo ottimo articolo](https://realpython.com/ref/stdlib/).

**che python stiamo usando?**

```python
  %python --version

  import sys
  print(sys.executable)
```



In [ ]:
!python --version
!python -V

In [ ]:
  import sys
  print(sys.executable, sys.version)

I seguenti package fanno tutti parte anch'essi della standard library:

In [ ]:
import random                               # serve per generare numeri casuali (importantissimo per generare dati sintetici)
import shutil                               # per operazioni su file e cartelle
from pathlib import Path                    # gestione ad oggetti di percorsi e file
from datetime import datetime, timedelta    # package per gestione temporale

Tutti gli import seguenti appartengono alla **Standard Library di Python**, quindi non richiedono alcuna installazione tramite `pip`.

| Import | Standard Library | Scopo principale |
|----------|:---:|------------------------------|
| `import random` | ✅ | Generazione di numeri casuali |
| `import shutil` | ✅ | Operazioni su file e directory (copia, spostamento, cancellazione, archiviazione, ...) |
| `from pathlib import Path` | ✅ | Gestione orientata agli oggetti di percorsi e file |
| `from datetime import datetime, timedelta` | ✅ | Gestione di date, orari e intervalli temporali |



In [ ]:
# Esempi:

# Numero casuale
n = random.randint(1, 100)
print(n)

# Oggetto percorso
cartella = Path("dati")
print(cartella)

# Data attuale
oggi = datetime.now()
print(oggi)

# Tra una settimana
fra_7_giorni = oggi + timedelta(days=7)
fra_7_giorni

I computer attuali NON sono in grado di generare numeri veramente casuali; generano numeri **pseudo-casuali*, cioè determinati da un **seme** iniziale. Questo limite è anche tuttavia un vantaggio perchè permette, grazie al seme, di garantire la fondamentale proprietà della **riproducibilità** --> un codice Python "riproducibile" si comporta esattamente allo stesso modo su ogni PC e ad ogni sessione.

In [ ]:
random.randint(1, 100)  # --> 82, 15,4

In [ ]:
random.seed(42)  # seme fisso => dati riproducibili a ogni esecuzione
                 # il valore del seme NON ha nessuna importanza, potete usare 1,1000,100000, la vostra data di nascita GGMMAAAA, ecc.

Allo stretto scopo di questo notebook la generazione dei dati sintetici supplisce alla mancanza di veri dati di produzione.

In generale, nelle attività sistemistiche di monitoraggio è essenziale poter generare dati sintetici (simulati) per **descrivere scenari non frequenti (per il test)**.

Con un linguaggio matematicamente corretto, prima di generare i dati, noi dobbiamo definire il **processo generativo dei dati**.

In [ ]:
import pandas as pd
df = pd.read_csv("clienti_banca_churn.csv")
display(df.head())
print(df.head())
print(type(df))
print("\n","Dimensioni del df: ",df.shape,"\n")
df.info()

---
**📝 Python generale**

I seguenti profili definiscono il comportamento dei tre server; cioè, in altre parole, definiscono il processo generativo dei dati.

---

In [ ]:
# Profilo di comportamento per ciascun host simulato (3): probabilità di errore, fascia di
# latenza "normale", fascia di latenza "lenta" e quanto spesso capita una richiesta lenta.
HOST_PROFILI = {
    "web01": {"err_prob": 0.01, "rt_base": (30, 90),  "rt_lento": (200, 500),  "prob_lento": 0.02},
    "web02": {"err_prob": 0.12, "rt_base": (80, 220), "rt_lento": (900, 2500), "prob_lento": 0.15},  # malato
    "web03": {"err_prob": 0.02, "rt_base": (40, 110), "rt_lento": (250, 600),  "prob_lento": 0.03},
}

print(type(HOST_PROFILI))   # un dizionario Python di 3 elementi multi-elemento, cioè un insieme di coppie <chiave, valore>,
                            # dove "chiave" è il nome del dato (metadato),
                            # e dove anche ogni riga ha un nome.
print(HOST_PROFILI['web01'])
HOST_PROFILI

**Cosa è `HOST_PROFILI`?**<br>
E' un **dizionario** che definisce il **profilo di comportamento di ciascun host simulato**: descrive *come* si comporta ogni server, così da poter fabbricare righe di log sintetiche ma realistiche.

La struttura è `nome_host → dizionario di parametri`, con quattro chiavi per host:

- `err_prob` — la probabilità che una richiesta sia un errore `5xx` (es. `0.12` = circa il 12% di errori)
- `rt_base` — la coppia `(min, max)` della latenza "normale" in ms
- `rt_lento` — la coppia `(min, max)` della latenza "della coda lenta" in ms
- `prob_lento` — quanto spesso una richiesta finisce nella fascia lenta invece che in quella normale

Dal punto di vista dei *type hint*, è un **dizionario annidato** (*nested dictionary*), a due livelli.

Infatti:
* le chiavi esterne sono stringhe ("web01", "web02", ...)
* i valori esterni sono altri dizionari nei dizionari interni

```text
HOST_PROFILI
│
├── web01
│   ├── err_prob   -> 0.01
│   ├── rt_base    -> (30, 90)
│   ├── rt_lento   -> (200, 500)
│   └── prob_lento -> 0.02
│
├── web02
│   ├── err_prob   -> 0.12
│   ├── rt_base    -> (80, 220)
│   ├── rt_lento   -> (900, 2500)
│   └── prob_lento -> 0.15
│
└── web03
    ├── err_prob   -> 0.02
    ├── rt_base    -> (40, 110)
    ├── rt_lento   -> (250, 600)
    └── prob_lento -> 0.03
```

Punto aperto: la distribuzione di probabilità dei tempi di risposta dei tre server. Vedi [questa chat](https://chatgpt.com/share/6a3a92b1-4780-83eb-8561-e5d739bb4eec) al fondo.

> ---
> **📌 Il processo generativo dei dati simulati**<br>
> `HOST_PROFILI` *è* la specifica del **processo generativo** dei dati simulati. Contiene i "veri" parametri del modello — `err_prob`, `prob_lento`, le fasce di latenza — da cui la successiva funziona `genera_righe` **campiona** per produrre ogni riga.
>
> Detto nel linguaggio probabilistico-statistico, è **un modello generativo parametrico** molto semplice. Ogni riga di log è un campione estratto da quelle distribuzioni:
> - lo `status` viene da una distribuzione di probabilità  [Bernoulli](https://it.wikipedia.org/wiki/Distribuzione_di_Bernoulli) (errore sì/no con probabilità `err_prob`) seguita da una categoriale sui codici,
> - la latenza `rt_ms` viene da una **mixture** di due uniformi — il regime "normale" `rt_base` e la "coda lenta" `rt_lento` — pesate da `prob_lento`. È, in piccolo, lo stesso schema di una [mixture distribution](https://en.wikipedia.org/wiki/Mixture_distribution).
>
> E qui c'è il dettaglio che rende l'esempio didatticamente onesto: quei parametri sono i **parametri veri ma latenti**. Lo strumento di monitoraggio che costruiamo nel notebook **non li conosce** — vede solo i campioni (le righe di log) e cerca di *risalire* alle proprietà del sistema stimando *error rate* e *p95* dai dati osservati.

> È esattamente **la situazione del troubleshooting reale**: i sistemisti non hanno accesso al "vero" `err_prob` di un server, **lo inferiscono dalle metriche storiche**. Noi, come autori del notebook, conosciamo `HOST_PROFILI`: questo ci dà la *ground truth* contro cui verificare che lo strumento funzioni: sappiamo che `web02` è davvero malato (`err_prob: 0.12`), quindi se lo strumento lo segnala come critico sta facendo il suo lavoro.
>
> Un paio di precisazioni per inquadrare bene lo strumento.
> - il `random.seed` fissato rende il processo generativo *riproducibile*: stesso seme, stessa realizzazione del campione — comodo per la didattica, ma è un campione particolare (lo stesso per tutti), non l'unico possibile.
> - il limite del modello: è deliberatamente **ingenuo** ([campioni i.i.d.](https://it.wikipedia.org/wiki/(Variabili_indipendenti_e_identicamente_distribuite), nessuna [correlazione](https://it.wikipedia.org/wiki/Correlazione_(statistica)) tra due variabili differenti, nessuna autocorrelazione, nessun pattern giorno/notte) perché il suo scopo è solo dare "materia prima" (dati di produzione) plausibile allo strumento, non modellare fedelmente il traffico reale. Più avanti, nel modulo 1.3, introdurremo tempo e correlazione.
>
> ---

---
**📝 Python generale**

Prima di definire la funzione `genera_righe` vogliamo capire cosa sono le **user-defined functions**.



---
La seguente funzione `genera_righe(host, profilo, n, t0)` legge questi parametri e per ogni riga "tira i dadi": con probabilità `err_prob` produce uno status `5xx`, e con probabilità `prob_lento` pesca la latenza da `rt_lento` anziché da `rt_base`. È per questo che `web02` ha valori volutamente più alti (`err_prob: 0.12`, `prob_lento: 0.15`, coda fino a 2500 ms): è il server **"malato"**, tarato perché lo strumento di monitoraggio lo segnali come critico su entrambe le metriche, mentre `web01` e `web03` restano sani.

`HOST_PROFILI` **non fa parte dello strumento** che stiamo insegnando a costruire. Serve solo a generare i file di log di esempio, per rendere il notebook autosufficiente e riproducibile (seme fisso → stessi dati a ogni esecuzione). Lo strumento vero — `parse_riga`, `leggi_eventi`, `aggrega`, … — non vede mai `HOST_PROFILI`: vede soltanto i log che ne risultano, esattamente come in produzione vedrebbe i log veri dei server.

Alcune **note tecniche** prima della definizione della funzione (per poterla capire):

* cosa significa davvero `t = t0`?

In Python una variabile non è una "scatola che contiene un valore", ma un'**etichetta** appiccicata a un oggetto. Quindi `t = t0` **non copia** nulla: mette semplicemente una seconda etichetta (`t`) sullo **stesso** oggetto `datetime` a cui punta `t0`.

A questo punto la domanda naturale è: se `t` e `t0` sono lo stesso oggetto, allora modificando `t` non modifico anche `t0`? Ed è qui che entra l'immutabilità. `datetime` è **immutabile**: `t += timedelta(seconds=3)` non può cambiare l'oggetto "sul posto". Quello che fa è **calcolare un oggetto nuovo** (`t0 + 3s`) e **spostare l'etichetta `t`** su quel nuovo oggetto. L'etichetta `t0` resta dov'era, sull'istante originale.

Concretamente:

```python
  t0 = datetime(2025, 6, 20, 14, 0, 0)  # 20.6.2025 14.00
  t = t0                          # t e t0: due etichette, stesso oggetto (le 14:00:00)
  t += timedelta(seconds=3)       # crea un NUOVO datetime (20.6.2025 14:00:03) e ci sposta SOLO t
  # t0 -> 2025-06-20 14:00:00   (intatto)
  # t  -> 2025-06-20 14:00:03   (avanzato)
```

È lo stesso meccanismo degli interi, che già conosci d'istinto: `x = 5; y = x; y += 1` lascia `x` a 5 e porta `y` a 6.

Con un oggetto **mutabile** la storia sarebbe diversa — ed è la classica trappola:

```python
  a = [1, 2]
  b = a            # stessa lista, due etichette
  b.append(3)
  # a ora è [1, 2, 3]!  -> modificando b ho modificato anche a, perché la lista è mutabile
```

Con una lista, `b = a` seguito da una modifica corromperebbe `a`. Con `datetime`, grazie all'immutabilità, `t = t0` è invece **sicuro**: per quanto faccia avanzare `t`, **`t0` non viene mai toccato**.

L'effetto netto nella funzione: `t0` è il **punto di partenza fisso**, `t` è l'**orologio del server che scorre**. Dato che `t0` non viene mai alterato, ogni host che generiamo riparte dallo stesso istante iniziale — `t0` vale `14:00:00` a ogni chiamata.

* `random.random()` che codominio ha?

`random.random()` restituisce un `float` nell'intervallo **`[0.0, 1.0)`**: zero **incluso**, uno **escluso**. In notazione: `0.0 <= x < 1.0`.



In teoria estrae da una distribuzione **uniforme** su quell'intervallo (ogni valore ugualmente probabile). Nella pratica, essendo un `float` a precisione finita, i valori possibili sono i multipli di 2⁻⁵³, ma per qualunque uso normale possiamo vederlo come su un continuo.

Il dettaglio degli estremi è esattamente ciò che rende corretto l'uso nella funzione `genera_righe`:

```python
  if random.random() < profilo["err_prob"]:   # es. err_prob = 0.12
```

Dato che `random.random() < soglia` è vera con probabilità pari alla `soglia` stessa, l'espressione **è vera circa il 12% delle volte**. E gli estremi tornano nei casi limite: con `err_prob = 0.0` la condizione è `x < 0.0`, **mai** vera (lo 0 è incluso nel codominio ma niente è minore di 0 → zero errori, giusto); con `err_prob = 1.0` è `x < 1.0`, **sempre** vera (perché 1.0 è escluso dal codominio → tutti errori, di nuovo giusto). Se gli estremi fossero inclusi/esclusi al contrario, questi due casi limite si comporterebbero in modo sbagliato.

Per completezza, le varianti imparentate hanno **codomini diversi**: `random.uniform(a, b)` copre `[a, b]` (estremi di norma inclusi), e `random.randint(a, b)` restituisce interi in `[a, b]` con **entrambi** gli estremi inclusi — ed è infatti il motivo per cui nel notebook `random.randint(1, 4)` può produrre anche `4`.

👉 **Il trucco**: **una probabilità vive in `[0, 1]`** e `random.random()` produce numeri (quasi) nello stesso intervallo: è questa coincidenza di domini che permette di usare un numero casuale per "decidere" se un evento con probabilità *p* deve accadere.

Il meccanismo è geometrico, ed è il modo più pulito di vederlo. Immagina il segmento da 0 a 1. Lo tagli in *p*: la fetta a sinistra è lunga *p*, quella a destra `1 − p`. Quando estrai un punto a caso **uniformemente** sul segmento, la probabilità che cada nella fetta di sinistra è esattamente la sua lunghezza, cioè *p*. E "cadere nella fetta di sinistra" è precisamente la condizione `random.random() < p`.

```
0                      p                          1
|———————————————————————|——————————————————————————|
   cade qui: prob. p          cade qui: prob. 1−p
   -> evento ACCADE            -> evento NON accade
```

La parola chiave è **uniforme**: funziona perché ogni punto del segmento è **equiprobabile**, quindi la probabilità di finire in una regione coincide con la sua lunghezza. Con una sorgente casuale non uniforme il trucco salterebbe — `< p` non darebbe più probabilità *p*.

Da qui discende anche la generalizzazione che usiamo nel notebook per scegliere *tra più esiti*: invece di un solo taglio, facciamo più tagli e guardiamo in quale fetta cade il punto. È esattamente cosa fa `random.choices(..., weights=[...])` dietro le quinte — partiziona `[0, 1)` (o `[0, somma_pesi)`) in fette proporzionali ai pesi e vede dove atterra l'estrazione. Il nostro `if random.random() < err_prob` è semplicemente il caso a due fette: "errore" / "non errore".

Una sottigliezza: il confronto è `<` (non `<=`), e l'estremo `1.0` è **escluso** dal codominio di `random.random()`. Questi due dettagli messi insieme fanno tornare i conti agli estremi — con `p = 0` l'evento non accade mai, con `p = 1` accade sempre — che è poi la verifica che la mappatura "numero casuale → decisione" sia davvero fedele alla probabilità *p*.

In [ ]:
a = [1, 2]
b = a            # stessa lista, due etichette
# b = a.copy()
b.append(3)
print(b)
print(a)
# a ora è [1, 2, 3]!  -> modificando b ho modificato anche a, perché la lista è mutabile

In [ ]:
def genera_righe(host, profilo, n, t0):   # parametri formali
    """Genera n righe di log sintetiche ma realistiche per UN host.

    NB: questa funzione serve solo a fabbricare i dati di esempio, così il
    notebook sarà autosufficiente e riproducibile. NON fa parte dello strumento
    di monitoraggio che stiamo costruendo: lo strumento vedrà soltanto i file
    di log prodotti, esattamente come in produzione vedrebbe i log veri dei server.

    Parametri
    ---------
    host    : str   -> nome del server, finirà nel campo `host=` di ogni riga
    profilo : dict  -> i parametri di comportamento dell'host (da HOST_PROFILI):
                       err_prob, rt_base, rt_lento, prob_lento
    n       : int   -> quante righe di log generare
    t0      : datetime -> istante di partenza del primo log

    Restituisce:
    -------
    list[str] -> le n righe di log, ciascuna nel formato `chiave=valore`.
    """
    righe = []          # accumulatore interno: qui mettiamo le righe man mano che le creiamo
    t = t0              # "orologio" interno; partiamo dall'istante iniziale t0.
                        # datetime è IMMUTABILE: t += ... crea un NUOVO oggetto datetime e lo
                        # riassegna a t, senza toccare t0. Così ogni host parte dallo
                        # stesso istante (t0 resta intatto per il prossimo host).

    for _ in range(n):  # ripetiamo n volte. `_` è la convenzione per "indice che non uso":
                        # ci interessa solo eseguire il corpo n volte, non il contatore.

        # --- 1) TIMESTAMP che avanza ---
        # Aggiungiamo da 1 a 4 secondi a ogni iterazione, per simulare richieste
        # ravvicinate ma con orari crescenti (un log reale non ha due righe identiche).
        t += timedelta(seconds=random.randint(1, 4))
        ts = t.strftime("%Y-%m-%dT%H:%M:%S")   # formato ISO 8601, es. "2025-06-20T14:00:03"
                                               # occorre infatti la conversione dal formato interno del package t a un formato standard

        # --- 2) CODICE DI STATO HTTP --- [nota 5]
        # random.random() estrae un float in [0.0, 1.0). Confrontarlo con una soglia
        # equivale a "tirare i dadi": è vero, in media, err_prob volte su 1 (per n → ∞) [nota 1].
        if random.random() < profilo["err_prob"]:
            # caso ERRORE: un 5xx (errore lato server) a caso fra questi tre
            status = random.choice([500, 502, 503])
        else:
            # caso OK: un 2xx/3xx. Il 200 è ripetuto 3 volte su 5 di proposito:
            # ripetere un valore nella lista è un modo semplice per "pesarlo"
            # (qui ~60% 200, 20% 201, 20% 304). Alternativa più esplicita:
            #   random.choices([200, 201, 304], weights=[3, 1, 1])
            status = random.choice([200, 200, 200, 201, 304])  # [nota 2]

        # --- 3) LATENZA (response time in ms) --- simulazione [nota 4]
        # Ogni tanto (prob_lento) una richiesta finisce nella "coda lenta", per
        # simulare i picchi di latenza che un buon monitoraggio deve scovare.
        if random.random() < profilo["prob_lento"]:
            # `*` [nota 3] spacchetta la tupla (min, max): se rt_lento = (900, 2500),
            # questo diventa random.randint(900, 2500).
            rt = random.randint(*profilo["rt_lento"])
        else:
            rt = random.randint(*profilo["rt_base"])   # latenza "normale"

        # --- 4) COMPOSIZIONE della riga ---
        # f-string: inseriamo i valori nel formato `chiave=valore` separato da spazi,
        # cioè lo STESSO formato che parse_riga() dovrà poi interpretare.
        righe.append(f"{ts} host={host} status={status} rt_ms={rt}") # righe è una lista, le liste hanno il metodo append

    return righe

**NOTE**:

**[1]**<br>
"in media, err_prob volte su 1" è un'affermazione **asintotica**: vale rigorosamente nel limite per n → ∞. Su un numero finito di righe la frazione osservata di errori sarà solo **vicina** a `err_prob`, non esattamente uguale.
È [**la legge dei grandi numeri**](https://it.wikipedia.org/wiki/Legge_dei_grandi_numeri):<br>
> la frequenza relativa di un evento converge alla sua probabilità man mano che le prove aumentano.<br>
Con `err_prob` = 0.12 e `n = 200`, non otteniamo 24 errori spaccati — ne otteniamo un numero intorno a 24, che oscilla a ogni realizzazione.

Quanto oscilla? Il numero di errori segue una binomiale di parametri n e p, con media np e deviazione standard √(n·p·(1−p)). Quindi la frazione osservata ha deviazione standard √(p(1−p)/n), che **scala come 1/√n: l'errore relativo si restringe, ma lentamente**. Nel nostro caso:

conteggio errori: media 200 × 0.12 = 24, deviazione √(200·0.12·0.88) ≈ 4.6 → tipicamente qualcosa come 24 ± 5
frazione: 0.12 ± 0.023 circa

Per questo, per inciso, il commento dice "in media" e non "esattamente": è la formulazione corretta. E spiega anche perché il random.seed fissato è importante per la didattica — inchioda una specifica realizzazione finita, così i numeri del notebook sono riproducibili. Ma resta una delle infinite realizzazioni possibili: con un seme diverso, web02 avrebbe un error rate leggermente diverso, sempre disperso attorno a 0.12.

NB. La stessa 1/√n dice che un error rate misurato su pochi campioni è inaffidabile. Tre richieste fallite su dieci non significano un p reale del 30% — l'incertezza su un campione così piccolo è enorme. La fiducia in una metrica di affidabilità cresce solo con il volume di richieste su cui la calcoli, ed è esattamente il tipo di cautela che separa una soglia di alert sensata da una che scatta al primo rumore statistico.

---
**[2]**<br>
`random.choice (seq)` **senza pesi è equiprobabile**, cioè estrae un elemento dalla sequenza in modo uniforme, cioè ogni posizione ha la stessa probabilità 1/len(seq).La sottigliezza — quella che conta per leggere il codice del notebook — è che **l'equiprobabilità è sulle posizioni, non sui valori distinti**. Se un valore compare più volte nella sequenza, "eredita" la probabilità di tutte le posizioni che occupa. Ed è esattamente il trucco usato qui.

---
**[3]**<br>
L'asterisco `*` è l'operatore di **unpacking** (spacchettamento): prende un iterabile — qui una tupla — e ne "versa" gli elementi come **argomenti separati** nella chiamata di funzione.

Il punto da cui partiamo è la differenza tra passare *un* oggetto e passare *più* argomenti. La funzione `random.randint` vuole **due** argomenti distinti, il minimo e il massimo:

```python
  random.randint(900, 2500)        # due argomenti separati: ok
```

Ma nel notebook l'intervallo è conservato come **una** tupla dentro al profilo:

```python
  profilo["rt_lento"]              # vale (900, 2500)  -> UN solo oggetto, la tupla
```

Se la passassimo così com'è, daremmo a `randint` un unico argomento (la tupla intera) invece di due numeri, ed esploderebbe:

```python
  random.randint(profilo["rt_lento"])     # random.randint((900, 2500)) -> ERRORE
```

L'asterisco risolve proprio questo: **apre la tupla e ne distribuisce gli elementi come argomenti separati**. Queste due righe diventano equivalenti:

```python
  random.randint(*profilo["rt_lento"])     # *(900, 2500)
  random.randint(900, 2500)                # ...è esattamente questo
```

In pratica `*` fa la traduzione `(900, 2500)` → `900, 2500`: da "un contenitore" a "i suoi contenuti, sparpagliati". Lo chiamiamo anche *argument unpacking*.

---
**[4]**<br>
Ecco una cella che isola **solo** il punto 3 (la latenza) e lo fa "girare a vuoto" per renderne visibile il meccanismo. La eseguiamo qui sotto così vediamo subito l'output reale (con seme fisso, quindi riproducibile).Ecco la cella, già verificata:

```python
import random

# Riprendiamo il profilo di un host "sofferente" (gli stessi valori di web02 in HOST_PROFILI)

# Definiamo un profilo per un host e solo rt
profilo = {"rt_base": (80, 220), "rt_lento": (900, 2500), "prob_lento": 0.15}

random.seed(0)   # seme fisso: i numeri qui sotto sono riproducibili

# --- Parte A: 15 estrazioni, per VEDERE come viene scelto il ramo -----------------
# è la parte della funzione 'genera_righe' relativa solo ad rt
# 'random.random() è qui chiamato 'dado'

print("estrazione |  dado  | ramo     | rt_ms")
print("-" * 42)
for i in range(1, 16):
    dado = random.random()                       # il "dado" in [0.0, 1.0)
    if dado < profilo["prob_lento"]:             # cade nella fetta "lenta" (lunga prob_lento)?
        rt = random.randint(*profilo["rt_lento"])
        ramo = "LENTO"
    else:
        rt = random.randint(*profilo["rt_base"])
        ramo = "normale"
    print(f"{i:>10} | {dado:.3f}  | {ramo:<8} | {rt}")

# --- Parte B: 10.000 estrazioni, per verificare che le proporzioni tornino ---------
n = 10_000                               # costante python che vale 10.000
n_lenti = somma_base = somma_lento = 0   # multi-assegnazione

# il solito ciclo (sino a 15):
for _ in range(n):
    if random.random() < profilo["prob_lento"]:
        n_lenti += 1
        somma_lento += random.randint(*profilo["rt_lento"])
    else:
        somma_base += random.randint(*profilo["rt_base"])

print(f"\nSu {n} richieste simulate:")
print(f"  lente : {n_lenti:>5}  ({n_lenti/n:.1%})   <- atteso ~{profilo['prob_lento']:.0%}")
print(f"  rt medio nel ramo 'normale' : {somma_base/(n - n_lenti):.0f} ms")
print(f"  rt medio nel ramo 'LENTO'   : {somma_lento/n_lenti:.0f} ms")
```

Output che otteniamo:

```
estrazione |  dado  | ramo     | rt_ms
------------------------------------------
         1 | 0.844  | normale  | 187
         2 | 0.040  | LENTO    | 1947
         3 | 0.486  | normale  | 157
         4 | 0.968  | normale  | 171
         5 | 0.583  | normale  | 135
         6 | 0.505  | normale  | 152
         7 | 0.140  | LENTO    | 1094
         8 | 0.618  | normale  | 144
         9 | 0.987  | normale  | 216
        10 | 0.983  | normale  | 117
        11 | 0.310  | normale  | 98
        12 | 0.899  | normale  | 164
        13 | 0.472  | normale  | 105
        14 | 0.354  | normale  | 160
        15 | 0.611  | normale  | 132

Su 10000 richieste simulate (in una certa esecuzione):
  lente :  1490  (14.9%)   <- atteso ~15%
  rt medio nel ramo 'normale' : 150 ms
  rt medio nel ramo 'LENTO'   : 1694 ms
```

Cosa leggiamo, e cosa conviene far notare in aula.

Nella **Parte A** vediamo il meccanismo riga per riga: il `dado` è il numero estratto, e ogni volta che cade sotto `0.15` (estrazioni 2 e 7) scatta il ramo `LENTO`, con una latenza nell'ordine delle migliaia di ms invece che delle centinaia. Su 15 estrazioni ne escono 2 lente: vicino al 15% atteso, ma con un campione così piccolo l'oscillazione è ampia — è di nuovo la cautela sui pochi campioni di cui parlavamo.

Nella **Parte B** la legge dei grandi numeri si vede all'opera: su 10.000 richieste la frazione di lente (14,9%) coincide quasi perfettamente con `prob_lento`. E le due medie separate — ~150 ms contro ~1694 ms — rendono evidente che stiamo campionando da **due regimi distinti**: è proprio questa "coda lenta" rara ma pesante che la media complessiva annegherebbe, e che invece la p95 dello strumento è progettata per scovare.

Aggiungiamo qui sotto un piccolo istogramma `matplotlib` delle latenze, così la separazione tra i due regimi diventa visiva oltre che numerica.

```python
import random
import numpy as np
import matplotlib.pyplot as plt

# Stesso profilo "sofferente" della demo precedente (gli stessi valori di web02)
profilo = {"rt_base": (80, 220), "rt_lento": (900, 2500), "prob_lento": 0.15}

random.seed(0)   # seme fisso: grafico riproducibile

# Generiamo le latenze con la STESSA logica del punto 3 di genera_righe
n = 5000
latenze = []
for _ in range(n):
    if random.random() < profilo["prob_lento"]:
        latenze.append(random.randint(*profilo["rt_lento"]))   # ramo LENTO (coda)
    else:
        latenze.append(random.randint(*profilo["rt_base"]))    # ramo normale

# Le due statistiche che vogliamo confrontare visivamente
media = np.mean(latenze)
p95 = np.percentile(latenze, 95)

# Istogramma
plt.figure(figsize=(10, 5))
plt.hist(latenze, bins=60, color="#4C72B0", edgecolor="white")
plt.axvline(media, color="orange", linestyle="--", linewidth=2, label=f"media = {media:.0f} ms")
plt.axvline(p95,  color="red",    linestyle="--", linewidth=2, label=f"p95 = {p95:.0f} ms")
plt.title(f"Distribuzione delle latenze ({n} richieste simulate)")
plt.xlabel("response time (ms)")
plt.ylabel("numero di richieste")
plt.legend()
plt.tight_layout()
plt.show()
```
Aggiungiamo l'istogramma. Lo eseguiamo qui così vediamo già il risultato, poi la cella è pronta da incollare in Colab.Ecco la cella, già verificata (in Colab usiamo `plt.show()`, qui abbiamo solo salvato il PNG per l'anteprima):

```python
import random
import numpy as np
import matplotlib.pyplot as plt

# Stesso profilo "sofferente" della demo precedente (gli stessi valori di web02)
profilo = {"rt_base": (80, 220), "rt_lento": (900, 2500), "prob_lento": 0.15}

random.seed(0)   # seme fisso: grafico riproducibile

# Generiamo le latenze con la STESSA logica del punto 3 di genera_righe
n = 5000
latenze = []
for _ in range(n):
    if random.random() < profilo["prob_lento"]:
        latenze.append(random.randint(*profilo["rt_lento"]))   # ramo LENTO (coda)
    else:
        latenze.append(random.randint(*profilo["rt_base"]))    # ramo normale

# Le due statistiche che vogliamo confrontare visivamente
media = np.mean(latenze)
p95 = np.percentile(latenze, 95)

# Istogramma
plt.figure(figsize=(10, 5))
plt.hist(latenze, bins=60, color="#4C72B0", edgecolor="white")
plt.axvline(media, color="orange", linestyle="--", linewidth=2, label=f"media = {media:.0f} ms")
plt.axvline(p95,  color="red",    linestyle="--", linewidth=2, label=f"p95 = {p95:.0f} ms")
plt.title(f"Distribuzione delle latenze ({n} richieste simulate)")
plt.xlabel("response time (ms)")
plt.ylabel("numero di richieste")
plt.legend()
plt.tight_layout()
plt.show()
```

Cosa rende evidente il grafico, e perché è il cuore del messaggio sulla latenza?<br>
Vediamo **due ammassi separati**: a sinistra il regime *normale* (80–220 ms), altissimo perché contiene l'85% delle richieste; molto più a destra, dopo un ampio vuoto, la *coda lenta* (900–2500 ms), fatta di barre basse perché quelle richieste sono rare.

Il punto didattico sono le due linee tratteggiate. La **media** (arancione, ~370 ms) cade nel vuoto tra i due ammassi: un valore che, preso da solo, sembra quasi accettabile e che **annega** completamente la coda. La **p95** (rossa, ~1918 ms) invece si colloca *dentro* la coda lenta, ed è ciò che permette di accorgersi che 1 richiesta su 20 sta soffrendo quasi 2 secondi. È esattamente perché lo strumento del Modulo 1.1 misura la p95 e non la media: la metrica giusta è quella che *non* lascia sparire i casi peggiori.

Se si vuole enfatizzare ancora di più la rarità della coda, possiamo aggiungere una seconda versione con l'asse y in scala logaritmica (le barre basse della coda diventano più leggibili), oppure annotare direttamente sul grafico le due regioni "normale" e "coda lenta".

---
**[5]**<br>
I codici di errore sono i **codici di stato HTTP**, lo standard con cui un server web comunica l'esito di ogni richiesta nella risposta. Sono i valori convenzionali del protocollo HTTP, quelli che vediamo realmente nei log di accesso. Li abbiamo scelti come campione rappresentativo dei casi tipici.

La logica è quella delle **classi** di codici HTTP, organizzate per la prima cifra:

- **2xx — successo.** La richiesta è andata a buon fine. `200 OK` è la risposta normale; `201 Created` indica che qualcosa è stato creato (tipico dopo una POST, es. un nuovo record).
- **3xx — redirezione.** `304 Not Modified` dice al client che la risorsa non è cambiata e può riusare la copia in cache: una risposta "buona" ed efficiente, molto comune nel traffico reale.
- **4xx — errore del client.** La richiesta è malformata o non consentita (es. `404 Not Found`, `403 Forbidden`). Nel notebook non li ho inclusi per tenere l'esempio focalizzato.
- **5xx — errore del server.** Qui il problema è lato server: `500 Internal Server Error` (errore generico), `502 Bad Gateway`, `503 Service Unavailable` (servizio sovraccarico o in manutenzione). Sono esattamente i codici che indicano un servizio in sofferenza.

Il motivo per cui abbiamo scelto **proprio questi** è didattico, e si lega direttamente a come `genera_righe` simula i due regimi:

```python
if random.random() < profilo["err_prob"]:
    status = random.choice([500, 502, 503])          # i 5xx = "errore del server"
else:
    status = random.choice([200, 200, 200, 201, 304]) # 2xx/3xx = "tutto ok"
```

La separazione rispecchia la definizione di error rate che lo strumento calcola: nel modulo 1.1, `aggrega` conta come errore le richieste con `status >= 500`. Quindi ci servivano due gruppi netti — un blocco di codici "sani" (2xx/3xx) e uno di codici "malati" (5xx) — con la soglia `500` a separarli puliti. I 4xx li abbiamo lasciati fuori proprio perché complicherebbero quella linea netta: un 4xx è un "errore", ma di norma è colpa del client, non un sintomo di un server in difficoltà, e mescolarlo confonderebbe la metrica di affidabilità che stiamo insegnando a misurare.

Il mix dei "sani" riprende il trucco della ripetizione di cui parlavamo: `200` tre volte su cinque perché la stragrande maggioranza del traffico reale è proprio `200 OK`, con qualche `201` e `304` a dare varietà plausibile.

Ecco una mini-tabella di riferimento dei codici HTTP più frequenti nei log (con significato e quando li incontriamo):




In [ ]:
import random

# Riprendiamo il profilo di un host "sofferente" (gli stessi valori di web02 in HOST_PROFILI)

# Definiamo un profilo per un host e solo rt
profilo = {"rt_base": (80, 220), "rt_lento": (900, 2500), "prob_lento": 0.15}

random.seed(0)   # seme fisso: i numeri qui sotto sono riproducibili

# --- Parte A: 15 estrazioni, per VEDERE come viene scelto il ramo -----------------
# è la parte della funzione 'genera_righe' relativa solo ad rt
# 'random.random() è qui chiamato 'dado'

print("estrazione |  dado  | ramo     | rt_ms")
print("-" * 42)
for i in range(1, 16):
    dado = random.random()                       # il "dado" in [0.0, 1.0)
    if dado < profilo["prob_lento"]:             # cade nella fetta "lenta" (lunga prob_lento)?
        rt = random.randint(*profilo["rt_lento"])
        ramo = "LENTO"
    else:
        rt = random.randint(*profilo["rt_base"])
        ramo = "normale"
    print(f"{i:>10} | {dado:.3f}  | {ramo:<8} | {rt}")

# --- Parte B: 10.000 estrazioni, per verificare che le proporzioni tornino ---------
n = 10_000                               # costante python che vale 10.000
n_lenti = somma_base = somma_lento = 0   # multi-assegnazione

# il solito ciclo (sino a 15):
for _ in range(n):
    if random.random() < profilo["prob_lento"]:
        n_lenti += 1
        somma_lento += random.randint(*profilo["rt_lento"])
    else:
        somma_base += random.randint(*profilo["rt_base"])

print(f"\nSu {n} richieste simulate:")
print(f"  lente : {n_lenti:>5}  ({n_lenti/n:.1%})   <- atteso ~{profilo['prob_lento']:.0%}")
print(f"  rt medio nel ramo 'normale' : {somma_base/(n - n_lenti):.0f} ms")
print(f"  rt medio nel ramo 'LENTO'   : {somma_lento/n_lenti:.0f} ms")

Quali codici di errore HTTP la funzione `genera_righe` utilizza?

Quelli ufficiali descritti nel seguente PDF, che contiene una loro **tassonomia**.

Per accedere ad un file PDF in Google Colab ci sono due modi:

In [ ]:
# prima modalità, vale per qualsiasi file si voglia caricare nella session storage;
# è alternativa alla modalità grafica dal menù verticale di sinistra
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()        # scegli il PDF dal tuo PC
    # ora è disponibile come /content/nome_file.pdf

A questo punto, nella *session storage* possiamo vedere il file PDF e fare doppio click sopra per renderizzarlo con un PDF reader esterno.

In modo analogo, possiamo fare doppio click su un'immagine nella *session storage*: in questo caso il rendering è interno a Google Colab.

Esiste poi una seconda modalità programmatica (non grafica) specifica solo per i PDF, che li renderizza in-line dentro il notebook stesso:

In [ ]:
%pip install pymupdf

In [ ]:
import fitz                                  # PyMuPDF
from IPython.display import Image, display

doc = fitz.open("/content/codici_http_riferimento.pdf")    # <-- metti qui il nome reale del tuo PDF
for pagina in doc:
    pix = pagina.get_pixmap(dpi=150)         # alza il dpi (es. 200) per più nitidezza
    display(Image(data=pix.tobytes("png")))

Proseguiamo nella pipeline di generazione dei log.

La seguente cella crea la cartella PULITA (lo scenario di TEST: dati senza imperfezioni).

---
**Alcune anticipazioni preliminari alla prossima cella di codice**<br>
`Path` è la **classe** per rappresentare <u>percorsi</u> di file e cartelle in modo orientato agli oggetti. Viene dal modulo `pathlib` della libreria standard (la famosa [***standard library di Python***](https://realpython.com/ref/best-practices/stdlib/?utm_source=notification_summary&utm_medium=email&utm_campaign=2026-06-22)), ed è **il modo moderno di lavorare col filesystem in Python** — al posto di manipolare i percorsi come semplici stringhe con il vecchio modulo `os.path`.

👉 L'idea di fondo: invece di trattare "logs/web01.log" come una stringa qualunque, lo incapsuliamo in un **oggetto `Path` percorso che ci mette a disposizione metodi e operatori pensati apposta**.

> Uno dei concetti cardine della [OOP](https://it.wikipedia.org/wiki/Programmazione_orientata_agli_oggetti) è che ogni **oggetto** (anche detto "istanza di classe") appartiene ad una classe (che ha dei metodi associati, disponibili solo per gli oggetti di quella classe specifica) e deve essere creato con un'operazione ad hoc che si chiama "**istanziazione**", nella quale in genere si definisce la configurazione dell'oggetto stesso. Una volta creato l'oggetto potremo manipolarlo con i metodi di classe con la *dot.notation*.<br>
> In taluni casi l'istanziazione è implicita in altre operazioni; ad esempio l'istruzione `a=1` istanzia l'oggetto `a` della classe <interi> e gli assegna il valore.

👉 Quando scriviamo `Path("logs")` non succede nulla sul disco: stiamo solo creando (cioè **istanziando**) l'oggetto, della classe percorsi, che rappresenta quello specifico percorso (`logs`). È un'azione separata dall'eventuale toccare il disco (creare, leggere, scrivere), che avviene solo quando chiamiamo i metodi opportuni.

La classe `Path` compone i percorsi con l'operatore `/`. Invece di concatenare stringhe a mano (rischiando di sbagliare gli slash), scriviamo `Path("logs") / "web01.log"` (che continua a non essere su disco, per ora). Tra l'altro `pathlib` usa il separatore giusto per il sistema operativo: "/" su Linux/Mac, "\ backslash" su Windows, quindi lo stesso codice funziona ovunque senza che ce ne preoccupiamo. Porta con sé i metodi sul filesystem. L'oggetto `Path` espone direttamente le operazioni, che nei notebook compaiono così:

`Path("logs").mkdir(exist_ok=True)` — crea la cartella in modo [idempotente](https://it.wikipedia.org/wiki/Idempotenza) grazie al parametro `exist_ok=True`<br>
per un generico oggetto 'percorso':<br>
`percorso.write_text(testo, encoding="utf-8")` — scrive un file in una sola chiamata in quel percorso<br>
`percorso.read_text(encoding="utf-8")` — lo legge<br>
`percorso.glob("*.log")` — elenca i file che corrispondono a un pattern in quel percorso<br>
`percorso.iterdir()` — scorre il contenuto di un percorso<br>
`percorso.open(...)` — apre il file (come `open()`, ma a partire dall'oggetto)

Ci sono poi gli attributi che danno i pezzi del percorso senza parsing manuale o automatico con funzioni `regex`:
- `percorso.name` (il nome del file, *web01.log*),
- `percorso.stem` (senza estensione, *web01*),
- `percorso.suffix` (l'estensione, *.log*),
- `percorso.parent` (la cartella che lo contiene).

Nel notebook 1.2, per esempio, la funzione `apri_testo` userà un proprio `percorso.suffix == ".gz"` per decidere se aprire il file con *gzip* — molto più pulito che cercare *".gz"* in una stringa.

Il contrasto con il vecchio approccio rende l'idea. Prima si scriveva:

In [ ]:
import os
percorso = os.path.join("logs", "web01.log")   # comporre
os.makedirs("logs", exist_ok=True)             # creare
nome = os.path.basename(percorso)              # estrarre il nome

Con `pathlib` diventa più leggibile e tutto "ruota" attorno all'oggetto:

In [ ]:
from pathlib import Path
percorso = Path("logs") / "web01.log"
Path("logs").mkdir(exist_ok=True)
nome = percorso.name

In sintesi: `Path` ci dà un **oggetto-percorso che compone i cammini in modo portabile** (tra OS  differenti) e ci offre, come **metodi**, tutte le **operazioni classiche su file e cartelle** — il che rende il codice <u>più leggibile e meno soggetto a errori</u> rispetto alle stringhe grezze. È esattamente per questo che nei notebook useremo `Path(...)` ovunque invece di passare nomi di file come semplici stringhe.

---

Ora possiamo davvero creare la cartella pulita e riempirla con i log:

In [ ]:
# ============================================================================
# CARTELLA PULITA — lo scenario di TEST: log "perfetti", senza imperfezioni.
# Generiamo un file di log per ciascun host (come in produzione, dove spesso
# ogni server/servizio scrive il proprio file, che poi un collettore aggrega).
# ============================================================================

# Path("logs") crea un OGGETTO percorso (pathlib): NON tocca ancora il disco, è
# solo una rappresentazione del percorso "logs". pathlib è il modo moderno di
# gestire file e cartelle, più pulito delle stringhe + os.path.
# .mkdir() crea fisicamente la cartella sul disco.
#   exist_ok=True -> se la cartella esiste già NON solleva un errore (utile perché
#                    il notebook può essere rieseguito più volte; senza, la seconda
#                    esecuzione fallirebbe con FileExistsError).
#   (per creare anche le eventuali cartelle padre mancanti si userebbe parents=True)
Path("logs").mkdir(exist_ok=True)

# datetime(anno, mese, giorno, ora, minuto, secondo): costruisce l'istante iniziale.
# È lo STESSO t0 per tutti gli host, quindi tutti i log partono dal medesimo momento
# (ricorda: genera_righe non modifica t0, perché datetime è immutabile).
t0 = datetime(2025, 6, 20, 14, 0, 0) # [nota 1]

# Iteriamo il dizionario dei profili. .items() restituisce coppie (chiave, valore),
# che spacchettiamo al volo in due variabili:
#   host -> il nome del server (la CHIAVE, una stringa: "web01", "web02", ...)
#   prof -> il suo profilo di comportamento (il VALORE, il dict dei parametri)
# (Iterare il dict "nudo", senza .items(), darebbe le sole chiavi.)
for host, prof in HOST_PROFILI.items():   # doppia variabile di controllo [nota 2]

    # Generiamo 200 righe di log per QUESTO host, secondo il suo profilo.
    # genera_righe restituisce una list[str]: una stringa per ogni riga di log.
    n = 200
    righe = genera_righe(host, prof, n, t0)   #  la chiamata della funzione
                                                #  con i parametri attuali

    # --- Scrittura del file ---
    # f"logs/{host}.log" costruisce il nome del file, es. "logs/web01.log".
    #   (Alternativa più idiomatica con pathlib:  Path("logs") / f"{host}.log")
    #
    # "\n".join(righe) unisce le righe inserendo un a-capo TRA una e l'altra:
    # per N righe mette N-1 a-capo, quindi l'ULTIMA riga resterebbe senza.
    # Il + "\n" finale aggiunge l'a-capo conclusivo, così OGNI riga (ultima
    # inclusa) termina correttamente -> file di testo "ben formato".
    #
    # .write_text(...) è la scorciatoia di pathlib che APRE il file, scrive
    # l'intera stringa e lo RICHIUDE, tutto in un'unica chiamata.
    #   encoding="utf-8" SEMPRE esplicito: non affidarsi al default del sistema
    #   operativo (su Windows può essere cp1252) per evitare sorprese tra ambienti.
    Path(f"logs/{host}.log").write_text("\n".join(righe) + "\n", encoding="utf-8")

**[Note]**<br>

**[1]**
`datetime(2025, 6, 20, 14, 0, 0)` costruisce un **oggetto `datetime`**, cioè un singolo istante nel tempo: rappresenta le **ore 14:00:00 del 20 giugno 2025**.

`datetime` è la classe del modulo `datetime` della libreria standard che combina **data e ora** in un unico valore. Scriverla in quel modo è una chiamata al suo costruttore, in cui passiamo i componenti dell'istante in ordine **dal più grande al più piccolo**:

```python
datetime(2025,  6,  20,  14,  0,  0)
#         anno  mese giorno ora min sec
```

Quindi, posizione per posizione: `2025` è l'anno, `6` il mese (giugno), `20` il giorno, `14` l'ora (in formato 24 ore, quindi le 2 del pomeriggio), `0` i minuti, `0` i secondi.

Un paio di cose utili da sapere. I primi tre argomenti — anno, mese, giorno — sono **obbligatori**; ora, minuti e secondi sono **opzionali** e valgono `0` se omessi. Per cui queste tre scritture danno lo stesso istante:

```python
datetime(2025, 6, 20, 14, 0, 0)
datetime(2025, 6, 20, 14)                       # min e sec sono 0 di default
datetime(year=2025, month=6, day=20, hour=14)   # esplicitando i nomi
```

I valori devono inoltre essere **sensati nel calendario**, altrimenti la costruzione fallisce: il mese sta in 1–12, l'ora in 0–23. Scrivere `datetime(2025, 13, 1)` solleva un errore, perché il mese 13 non esiste.

Nel notebook questo oggetto è il nostro `t0`, l'**istante di partenza** da cui far avanzare l'orologio dei log. Lo abbiamo scelto noi, fisso, così la generazione è riproducibile: **ogni host comincia a registrare richieste a partire dalle 14:00:00 di quel giorno**, e da lì `genera_righe` aggiunge i secondi via via.

Una precisazione: questo `datetime` è **"naive"**, cioè non porta con sé alcun fuso orario — è un istante "puro", senza l'informazione se siano le 14:00 a Roma o a Londra. Per i dati simulati va benissimo, ma è esattamente il nodo che affronteremo nel modulo 1.3, quando dovremo correlare log che arrivano da sorgenti con fusi diversi e quel "naive" non basterà più.

**[2]**<br>
`dizionario.items()` è un metodo dei dizionari Python che restituisce una vista (`dict_items`) contenente tutte le **coppie chiave-valore**.<br>
Il dizionario `HOST_PROFILI`,  come noto, è annidato, dunque il ciclo deve iterare su due chiavi:
```python
  for host, prof in HOST_PROFILI.items():  
    print(host,prof)
```

In [ ]:
# cella di test del loop (della celle precedente)
for host, prof in HOST_PROFILI.items():
  print(host,prof)

In [ ]:
type(HOST_PROFILI.items)

In [ ]:
?datetime

Due punti:
* il perché del + "\n" finale: il join mette i separatori tra gli elementi, quindi l'ultima riga resterebbe senza terminazione
* `encoding="utf-8"` esplicito non è pedanteria — su Windows il default può essere `cp1252`, e affidarvisi è proprio la causa dei bug di encoding che vedremo nel modulo 1.2.

L'alternativa: `pathlib Path("logs") / f"{host}.log"`, che è il modo più idiomatico di comporre i percorsi.

Ora creiamo la **cartella reale (di produzione)**, con i dati PULITI della cartella precedente, e poi facciamo alcune note:

In [ ]:
# 2) Cartella REALE (la PRODUZIONE: dati puliti + imperfezioni) ---------------------

# rmtree con ignore_errors=True: partiamo sempre da una cartella pulita, anche se
# il notebook viene rieseguito più volte (operazione idempotente, niente errori).

shutil.rmtree("logs_reali", ignore_errors=True)
Path("logs_reali").mkdir()
for p in Path("logs").glob("*.log"):             # glob("*.log") -> solo i file di log...
    shutil.copy(p, Path("logs_reali") / p.name)  # ...copiati PULITI, così come sono nella cartella "reale"

Alcune note per la comprensione della cella precedente:

- **`shutil`** (*shell utility*) è il modulo della libreria standard per operazioni "di alto livello" su file e cartelle (copiare, spostare, eliminare interi alberi). `pathlib` gestisce un percorso alla volta; `shutil` lavora su intere strutture.

- **`shutil.rmtree`** cancella una cartella **e tutto il suo contenuto in modo ricorsivo** (sottocartelle e file inclusi). È un'operazione **distruttiva e definitiva** — niente cestino — quindi va usata con attenzione sul percorso giusto. Nota in più: `ignore_errors=True` copre *anche* il caso della **prima** esecuzione, quando `logs_reali` non esiste ancora (senza, mancando la cartella, solleverebbe un errore); così funziona sia al primo giro sia ai successivi.

- Il **`mkdir()` qui è senza `exist_ok=True`** — al contrario della cartella `logs`. È voluto: `rmtree` ha appena garantito che la cartella non c'è più, quindi la ricreiamo pulita; se per qualche motivo esistesse ancora, `mkdir` solleverebbe un errore, fungendo da piccolo controllo di sanità.

- **`glob` restituisce un iteratore "pigro" (*lazy*) di oggetti `Path`** (non una lista): valuta i file man mano. Per questo `p` è già un `Path`, e puoi usarne `p.name` e passarlo direttamente a `shutil.copy`. Il `*` è un *wildcard* (qualsiasi sequenza di caratteri) e `glob` cerca **solo nella cartella indicata**, non nelle sottocartelle (per quello servirebbe `rglob`).

- **`shutil.copy(sorgente, destinazione)`** copia il contenuto del file **e i bit di permesso** (non i timestamp: per quelli c'è `copy2`; per il solo contenuto `copyfile`). Qui la destinazione `Path("logs_reali") / p.name` riusa lo **stesso nome del file**, così `logs/web01.log` diventa `logs_reali/web01.log`.

> **L'intento d'insieme di questa cella**: la cartella "reale" si costruisce partendo da una **copia dei log puliti** come base, su cui poi (nella cella successiva) vengono *aggiunte* le righe sporche. I dati puliti restano la base, le imperfezioni sono uno strato sopra — mantenendo la riproducibilità.

...e ora iniettiamo la "realtà": righe che il parser INGENUO non si aspetta.
Ognuna rappresenta un caso di sporcizia tipico dei log veri.<br>
Creiamo una lista di nome `righe_sporche` di 4 elementi per il server 2 (*web02*), quello "malato", che accodiamo alla parte di log "sana" (generata dalle celle precedenti):

In [ ]:
righe_sporche = [
    "2025-06-20T15:01:10 host=web02 status=500",            # manca il campo rt_ms
    "2025-06-20T15:01:11 host=web02 status=200 rt_ms=NaN",  # rt_ms presente ma non numerico
    "QUESTA RIGA E' ROTTA",                                 # nessun campo chiave=valore
    "2025-06-20T15:01:13 status=200 rt_ms=55",              # manca il campo host
]

# Modalità "a" = APPEND: aggiunge in coda a web02.log SENZA sovrascrivere i dati
# puliti appena copiati (la modalità "w" li cancellerebbe). [nota 1]

with open("logs_reali/web02.log", "a", encoding="utf-8") as f:
    for r in righe_sporche:
        f.write(r + "\n")

# lavorando su un singolo percorso (e non su una cartella) possiamo nuovamente usare il package 'pathlib' e la classe 'Path'
Path("logs_reali/README.txt").write_text("Cartella dei log applicativi.\n", encoding="utf-8")

print("Creati:", [p.name for p in sorted(Path("logs").iterdir())],
      "in logs/ e", [p.name for p in sorted(Path("logs_reali").iterdir())], "in logs_reali/")


**Note**<br>
**[1]**:<br>
Il modo più comune e sicuro per <u>leggere</u> un file in Python è utilizzare il costrutto `with`.

```python
with open("dati.txt", "r") as f:
    contenuto = f.read()

print(contenuto)
```

**Modalità di accesso (il parametro `mode`) di `open()`**

Il secondo parametro di `open()` specifica **come** si vuole aprire il file.

| Modalità | Significato |
|-----------|-----------|
| `"r"` | Lettura (Read) |
| `"w"` | Scrittura (Write) |
| `"a"` | Accodamento (Append) |
| `"x"` | Creazione esclusiva |
| `"r+"` | Lettura e scrittura |
| `"w+"` | Lettura e scrittura (svuota il file) |
| `"a+"` | Lettura e scrittura in append |

**[2]**:<br>
??? Un file ESTRANEO finito per errore nella cartella dei log: serve a verificare che lo strumento legga solo i veri file di log e ignori il resto (lo vedremo con glob). ??

**Check intermedio**: diamo ora un'occhiata al formato (dati puliti) e alla coda "sporca" della produzione che abbiamo creato nelle celle precedenti:

In [ ]:
print("--- logs/web02.log (prime 5 righe, dati puliti) ---")
print("\n".join(Path("logs/web02.log").read_text(encoding="utf-8").splitlines()[:5]))   # chaining di metodi

print("\n--- logs_reali/web02.log (ultime 6 righe: 2 pulite + 4 sporche) ---")
print("\n".join(Path("logs_reali/web02.log").read_text(encoding="utf-8").splitlines()[-6:]))


**Cosa fa questa riga**

Annida due strutture: join e chaining.

Al primo livello Python valuta l'espresisone dall'interno all'esterno, e dunque esegue per primo il parametro della join, al quale poi applica la join.

Chaining è fatto sul parametro della `join`: `Path("logs_reali/web02.log").read_text(encoding="utf-8").splitlines()[-6:]`

Il *chaining* --> è una **catena** di chiamate: ogni metodo agisce sul risultato del precedente. Python la valuta **da sx a dx**, quindi conviene leggerla così. L'effetto netto è uno solo: **stampare le prime 5 righe del file**. Vediamola pezzo per pezzo.

**1. `Path("logs/web02.log")`**
Creiamo un oggetto percorso che rappresenta il file. Qui non leggiamo ancora nulla dal disco: costruiamo solo il "riferimento" al file su cui lavoreremo.

**2. `.read_text(encoding="utf-8")`**
Leggiamo l'**intero** contenuto del file e lo otteniamo come **un'unica stringa**, comprensiva dei caratteri di a-capo `\n`. Qualcosa come:

```python
"2025-06-20T14:00:03 host=web02 status=200 rt_ms=47\n2025-06-20T14:00:06 host=web02 status=500 rt_ms=1203\n...(tutto il file)..."
```

Dichiariamo `encoding="utf-8"` sempre, per non dipendere dal default del sistema (lo stesso punto del modulo 1.2).

**3. `.splitlines()`**
Spezziamo quella stringa unica in una **lista di stringhe, una per riga**, **rimuovendo** i caratteri di a-capo:

```python
['2025-06-20T14:00:03 host=web02 status=200 rt_ms=47',
 '2025-06-20T14:00:06 host=web02 status=500 rt_ms=1203',
 '...', ...]   # tante stringhe quante sono le righe del file
```

**4. `[:5]`**
È uno **slice**: teniamo solo i **primi 5 elementi** della lista (indici 0–4), cioè le prime 5 righe. È un'operazione sicura: se il file avesse meno di 5 righe, restituirebbe semplicemente quelle che ci sono, senza errori.

```python
[riga0, riga1, riga2, riga3, riga4]
```

Valuata l'espressione interna (il parametro della join), Python esegue la join esterna:

**5. `"\n".join(...)`**
Riuniamo quelle 5 righe in **una sola stringa**, inserendo un a-capo **tra** una e l'altra (rimettiamo i `\n` che `splitlines` aveva tolto):

```python
"riga0\nriga1\nriga2\nriga3\nriga4"
```

**6. `print(...)`**
Stampiamo quella stringa: i `\n` interni mandano a capo, quindi vediamo le 5 righe incolonnate.

Due dettagli che in aula conviene far notare. Il primo: perché spezzare e poi rincollare? Perché `read_text` ci dà *tutto* il file in un colpo, ma a noi serve solo una **sbirciatina** (le prime righe); spezzare → tagliare a 5 → ricomporre è l'idioma per ottenere la "testa" di un testo. Il secondo: usiamo `splitlines()` e non `split("\n")` perché `splitlines()` è più robusto — riconosce i diversi terminatori di riga (`\n`, `\r\n`) e non lascia una stringa vuota in coda quando il file finisce con un a-capo.

Un'ultima nota di coerenza col corso: qui `read_text()` carica l'intero file in memoria, e va benissimo perché è un file piccolo e ci serve solo un assaggio. Su un log enorme non lo faremmo: useremmo lo **streaming** con il generatore `leggi_righe` del modulo 1.2, sbirciando le prime righe con `itertools.islice` invece di caricare tutto.

# L'obiettivo: dal log al report di affidabilità

Ora che abbiamo prodotto i log dei server, vogliamo accederli e produrre, per ciascun host, un piccolo
**report di affidabilità**. Questo è il compito; lo affronteremo **due volte**:
* prima come
*script* lineare (*as-is*),
* poi ricostruito in forma *idiomatica* (lo strumento di monitoraggio ben costruito).

L'obiettivo
non cambia — cambia il modo di scriverlo.

In sintesi, il flusso è:

1. **Scorrere** tutti i file nella cartella dei log.
2. Per ogni riga (di ogni file) **estrarre** i campi `host`, `status` e `rt_ms` dal formato `chiave=valore`.
3. **Accumulare** per ogni host il numero di richieste, il numero di errori (`status >= 500`)
   e l'elenco delle latenze.
4. **Calcolare** i due KPI: l'`error_rate` (percentuale di errori) e il `p95` della latenza.
5. **Stampare** il report e segnalare gli host che superano le soglie.

# La versione AS-IS

Ecco come "potrebbe" essere scritto il vostro script oggi: **un unico blocco lineare**. Apre i file, spezza le righe,
accumula i conteggi in **dizionari paralleli** (`richieste`, `errori`, `latenze`), calcola i
KPI e stampa il report — tutto mescolato. Le soglie sono **numeri magici** sparsi nel codice,
i percorsi sono **cablati**.

È il modo in cui uno script *nasce*. Lo facciamo girare sui dati di **test** puliti  (`logs/`).

In [ ]:
import os

CARTELLA = "logs"   # <-- percorso cablato nel codice

# dizionari "paralleli": programmazione fragile
richieste = {}      # host -> numero di richieste
errori = {}         # host -> numero di 5xx
latenze = {}        # host -> lista dei rt, per il p95

# codice spaghetti (intrecciato)
for nome_file in os.listdir(CARTELLA):                  # legge TUTTO ciò che trova nella cartella
    percorso = os.path.join(CARTELLA, nome_file)
    f = open(percorso)                                  # niente context manager, niente encoding
    for riga in f:
        parti = riga.split()
        host = None
        status = None
        rt = None
        for token in parti[1:]:                         # parti[0] è il timestamp
            chiave, valore = token.split("=")           # assume sempre esattamente un '='
            if chiave == "host":
                host = valore
            elif chiave == "status":
                status = int(valore)                    # assume sempre numerico
            elif chiave == "rt_ms":
                rt = int(valore)
        if host not in richieste:                       # inizializzazione "a mano" dei 3 dizionari
            richieste[host] = 0
            errori[host] = 0
            latenze[host] = []
        richieste[host] += 1
        if status >= 500:
            errori[host] += 1
        latenze[host].append(rt)
    f.close()

# calcolo dei KPI e report: anch'essi dentro lo stesso flusso lineare
print("=== Report host (AS-IS) ===")
for host in richieste:
    n = richieste[host]
    err_rate = errori[host] / n
    ordinate = sorted(latenze[host])
    p95 = ordinate[int(len(ordinate) * 0.95)]           # indice del percentile calcolato "al volo"
    print(f"{host} | richieste: {n} | error_rate: {round(err_rate, 3)} | p95_ms: {p95}")
    if err_rate > 0.05:                                 # numeri magici: 0.05 e 800
        print("   ATTENZIONE: error rate alto")
    if p95 > 800:
        print("   ATTENZIONE: latenza p95 alta")


**Interpretazione del report**

L'output è il report di affidabilità prodotto dallo script as-is, una riga per host. Lo leggiamo così:

**`web02` è il server in sofferenza**, e lo script lo segnala su entrambe le metriche. Ha un `error_rate` di **0.095** (il 9,5% delle richieste è un errore `5xx`), ben oltre la soglia di 0.05 → "error rate alto". E una `p95` di **1646 ms**: una richiesta su venti si avvicina ai ~2 secondi, oltre la soglia di 800 ms → "latenza p95 alta". È esattamente l'host che avevamo tarato come "malato" in `HOST_PROFILI`, quindi lo strumento sta facendo il suo lavoro.

**`web01` e `web03` sono sani.** Error rate a 0.0 e 0.01, p95 a 89 e 109 ms: ben sotto entrambe le soglie, nessun allarme.

Un paio di osservazioni:
* tutti i log hanno **200 richieste** perché è il numero di righe che `genera_righe` produce per host. E i numeri non sono perfettamente identici ai parametri di `HOST_PROFILI` — `web02` ha `error_rate` 0.095 e non esattamente 0.12 — perché sono **stime su un campione finito** (200 richieste), disperse attorno al valore "vero": è di nuovo la legge dei grandi numeri di cui parlavamo, e il motivo per cui una metrica va sempre letta tenendo conto del volume su cui è calcolata.
* il report **funziona** e ha senso — su questi dati puliti. È proprio ciò che rende efficace il contrasto con la cella successiva: la stessa logica, messa davanti a `logs_reali/`, si spezzerebbe alla prima riga malformata (una delle 4 righe finali di *web02*)

**Lo script funziona!** — e su `web02` scattano correttamente gli allarmi. Ma guardiamolo con gli
occhi di chi lo dovrà mantenere e far girare in produzione:

- **Percorso cablato.** Per cambiare cartella si edita il codice. Non è riusabile su un altro
  ambiente senza modificarlo.
- **Tutto intrecciato.** Lettura, parsing, validazione, aggregazione, calcolo del percentile,
  report e *alerting* vivono nello stesso ciclo. Non puoi toccare un pezzo senza rischiare di danneggiare gli altri. Studi seri dicono che, con codice "spaghetti" 1 correzione di baco su 5 introduce involontariamente un altro baco (senza accorgersene)
- **Dizionari paralleli.** `richieste`, `errori`, `latenze` vanno tenuti allineati a mano: una
  modifica e si disallineano.
- **Soglie** `0.05`, `800`, `0.95`, `500` non hanno nome né un punto unico dove
  cambiarli.
- **Nessuna separazione di "concern" (responsabilità).** ad esempio, il calcolo è inseparabile dall'I/O: non puoi verificarlo in
  isolamento.
- **Riuso, nessun test.** Non ci sono funzioni riusabili e testabili separatamente.
- **Nessuna difesa sui dati errati** (le cosiddette "guardie", controlli <u>intermedi</u> sulla corretteza dei dati intermedi). Assume che *ogni* riga sia perfetta. In test lo è. In
  produzione, no — come vediamo subito qui sotto.

La seguente cella dimostra le **fragilità** del parser *as-is*: mostra in modo concreto e controllato come e perché lo script lineare si rompe (o, peggio, sbaglia in silenzio) quando incontra le righe imperfette del mondo reale.

In [ ]:
# Cosa fa il parser ingenuo di fronte alle righe "vere"? Lo proviamo riga per riga,
# protetti da un try/except solo per non interrompere il notebook.
righe_problematiche = [
    "2025-06-20T15:01:10 host=web02 status=500",            # manca rt_ms
    "2025-06-20T15:01:11 host=web02 status=200 rt_ms=NaN",  # rt_ms non numerico
    "QUESTA RIGA E' ROTTA",                                 # niente chiave=valore
]

for riga in righe_problematiche:
    print(f"\nRiga: {riga!r}")
    try:
        parti = riga.split()
        host = status = rt = None
        for token in parti[1:]:
            chiave, valore = token.split("=")              # <-- esplode se manca '='
            if chiave == "host":   host = valore
            elif chiave == "status": status = int(valore)
            elif chiave == "rt_ms":  rt = int(valore)      # <-- esplode se non numerico
        # se siamo arrivati qui senza eccezioni, controlliamo cosa abbiamo ottenuto
        print(f"  risultato: host={host} status={status} rt={rt}")
        if rt is None:
            print("  >>> nessun errore qui, ma rt=None: un dato SBAGLIATO in silenzio,")
            print("      che farà detonare il calcolo del p95 a valle (sorted con None).")
    except Exception as e:
        print(f"  >>> lo script si INTERROMPE: {type(e).__name__}: {e}")


Tre righe sporche, **tre comportamenti diversi**.

**Riga 1 — `host=web02 status=500` (manca `rt_ms`)** → il fallimento **silenzioso**. Nessuna eccezione: il parser "passa", ma restituisce `rt=None`. È il caso più insidioso, perché sul momento sembra tutto a posto — il dato monco però detonerà più a valle, quando la `p95` proverà a fare `sorted(...)` su una lista che contiene un `None`.

**Riga 2 — `rt_ms=NaN` (valore non numerico)** → il fallimento **rumoroso**. `int("NaN")` solleva `ValueError: invalid literal for int()`. Lo script si interromperebbe qui.

**Riga 3 — `QUESTA RIGA E' ROTTA` (nessun `chiave=valore`)** → di nuovo un crash rumoroso, ma per una causa diversa: `token.split("=")` su un token senza `=` restituisce un solo elemento, e lo spacchettamento `chiave, valore = ...` fallisce con `ValueError: not enough values to unpack (expected 2, got 1)`.

Messaggio: **due righe su tre faranno crashare lo script, una lo farà sbagliare in silenzio** — e quest'ultima è la più pericolosa, perché non lascia traccia finché il danno non si propaga altrove. Sono esattamente i tre casi che `parse_riga`, nella **prossima versione idiomatica**, neutralizzerà restituendo `None` in modo pulito ed esplicito: niente crash, niente dati monchi che passano inosservati.



# Verso lo strumento (la parte idiomatica del notebook)

---
**📌Nota sul termine "idioma/idiomatico"**

<u>Idiomatico</u>: viene da idioma, e a sua volta dal greco idíōma, che significa "peculiarità, espressione propria" (dalla radice ídios = "proprio, particolare" — la stessa di "idiota", che in origine voleva dire semplicemente "persona privata, comune", e di "idiosincrasia").

Il senso originario è quello linguistico: **un idioma è un modo di esprimersi proprio di una lingua**, una frase fatta che i parlanti madrelingua usano naturalmente ma che non si spiega traducendo parola per parola. "Piove a catinelle", "in bocca al lupo": sono idiomatiche perché è così che si dice in italiano, anche se la costruzione, presa alla lettera, non sarebbe l'unica logicamente possibile. Una frase grammaticalmente corretta ma costruita "alla maniera di un'altra lingua" suona innaturale a un madrelingua, anche se si capisce.

👉 Da qui il termine **è passato per analogia alla programmazione**, e questo è il senso che usiamo nel corso. Il codice idiomatico è il codice scritto "come si dice" in quel linguaggio: **il modo naturale, riconosciuto dalla comunità**, che sfrutta gli strumenti propri del linguaggio. Non è solo codice corretto — è codice che un esperto di quel linguaggio riconosce come "scritto bene, alla sua maniera".

---
Ricostruiamo la stessa attività di prima (dello script *as-is*) separando le responsabilità, **un concern ("responsabilità") alla volta**.

Ogni pezzo avrà un solo compito e un nome.

La logica starà in funzioni **pure**; l'I/O ai **bordi**.

Gli errori saranno **gestiti**.

Le soglie diventeranno **configurazione** in file esterno (e non cablati nel software).

Alla fine i pezzi si comporranno in un unico strumento — con CLI (terminale), gestione errori, *exit code* e test.

Ecco una tabellina di sintesi:

| Concern (responsabilità) | Difetto as-is che risolve |
|---|---|
| Funzioni pure: un solo compito e un nome | *Tutto intrecciato* e *nessun riuso / nessun test* |
| I/O ai bordi | *Calcolo inseparabile dall'I/O* (e abilita i test) |
| Gestione esplicita degli errori | *Nessuna difesa sui dati errati* (le tre righe che esplodono) |
| Soglie come configurazione esterna | *Numeri magici dentro il codice* (`0.05`, `800`, `500`…) |
| `dataclass` per i record | *Dizionari paralleli* da allineare a mano |
| CLI ed *exit code* | *Percorso cablato* nel codice |

## Logging

Ecco la **prima cella idiomatica**: la configurazione del  logging, che appartiene al concern "diagnostica", e che prepariamo qui all'inizio, pronta per tutte le celle successive:

In [ ]:
import logging                           # logging strutturato (warning/info) al posto di print: qui per segnalare le righe scartate, separando la diagnostica dall'output

# Un logger configurato una volta sola. logging (non print) separa la DIAGNOSTICA
# (warning, info) dall'OUTPUT vero, e permette di filtrare per livello o dirottare su file.
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)  # configurazione del logging (per tutti i logger che creeremo)
logger = logging.getLogger("monitor")                                                    # costruttore (del singolo logger al quale possiamo dare un nome)

Queste due righe **preparano il sistema di logging** prima di usarlo: la prima ne imposta il comportamento globale, la seconda crea l'oggetto con cui scriveremo i messaggi. Le smontiamo una alla volta.

```python
  logging.basicConfig(...)` # la configurazione globale
```

`basicConfig` configura, **una volta per tutta l'applicazione**, *come* devono essere trattati i messaggi di log: a quale soglia di importanza, in che formato, verso quale destinazione. I tre argomenti che passiamo:

* **`level=logging.INFO`** — la **soglia minima** di importanza: vengono mostrati solo i messaggi di livello `INFO` o superiore. I livelli di logging sono ordinati per gravità crescente:

```text
  DEBUG  <  INFO  <  WARNING  <  ERROR  <  CRITICAL
```

Con la soglia a `INFO`, i messaggi `DEBUG` (più verbosi, di dettaglio) vengono **soppressi**, mentre `INFO`, `WARNING`, `ERROR` e `CRITICAL` passano. È un **filtro**: alziamo la soglia a `WARNING` se vogliamo solo gli avvisi e oltre, la abbassiamo a `DEBUG` quando dobbiamo investigare a fondo. Nel nostro strumento usiamo `logger.info(...)` per il riepilogo e `logger.warning(...)` per le righe scartate, quindi con `INFO` li vediamo entrambi.

* **`format="%(levelname)s %(message)s"`** — la **forma** di ogni riga di log. È un template con dei segnaposto che `logging` sostituisce: `%(levelname)s` diventa il nome del livello (`WARNING`, `INFO`…) e `%(message)s` il testo del messaggio. Con questo formato, `logger.warning("riga scartata")` produce:

```text
  WARNING riga scartata
```

Senza il `format`, la riga avrebbe l'aspetto di default del modulo. Volendo, potremmo arricchirlo: per esempio `format="%(asctime)s %(levelname)s %(message)s"` antepone il timestamp a ogni messaggio.

Il timestamp è messo in automatico dal logger, non dobbiamo configurarlo.

* **`force=True`** — questo serve se siamo in VSC/Colab, e merita una spiegazione. Normalmente `basicConfig` ha effetto **solo la prima volta** che viene chiamato: se il logging risulta già configurato, le chiamate successive vengono **ignorate** in silenzio. In un notebook questo è un problema, perché l'ambiente (o una cella eseguita prima) può aver già configurato il logging a nostra insaputa, e allora la nostra `basicConfig` non avrebbe effetto — il formato e il livello che impostiamo verrebbero ignorati. `force=True` dice: "**ripristina e riapplica** questa configurazione comunque", scartando quella eventualmente presente. È ciò che ci garantisce di vedere davvero i messaggi nel formato che vogliamo, anche rieseguendo la cella più volte.

```python
logger = logging.getLogger("monitor")` # il nostro logger
```

Questa riga **ottiene un logger con un nome**, `"monitor"`, e lo mette nella variabile `logger`. Da qui in poi scriviamo i messaggi chiamando i suoi metodi: `logger.info(...)`, `logger.warning(...)`.

Un paio di punti che vale la pena chiarire:
* `getLogger("monitor")` non *crea* ogni volta un oggetto nuovo: `logging` tiene un registro dei logger per nome, quindi chiamarlo con lo stesso `"monitor"` in celle diverse restituisce **sempre lo stesso** logger. Questo è comodo: possiamo "recuperarlo" da qualunque cella senza passarcelo in giro.
* il **nome** serve a identificare la sorgente dei messaggi. In un programma piccolo è quasi estetica, ma in un'applicazione reale composta da più moduli, dare a ciascuno il proprio logger (`getLogger(__name__)` è la convenzione tipica) permette di capire *da dove* arriva ogni messaggio e di regolare il livello modulo per modulo. Qui scegliamo `"monitor"` come etichetta del nostro strumento.

**Prché due righe separate?**

Per la divisione di compiti, perché riflette il principio del modulo. `basicConfig` è la **configurazione globale** — riguarda l'intero sistema di logging e si fa una volta sola. `getLogger` ci dà lo **strumento di lavoro** — l'oggetto che usiamo per emettere i messaggi. La prima decide le *regole* (soglia, formato, destinazione), il secondo è il *canale* attraverso cui scriviamo. Sono due cose distinte, ed è corretto vederle come tali — esattamente come, poco fa, abbiamo riconosciuto che la configurazione del logging e la definizione delle `dataclass` sono concern separati.

## Modellare i dati: `dataclass` + type hints

Lo script *as-is* teneva i dati in **dizionari paralleli**: più dizionari separati che
descrivono gli stessi oggetti, tenuti in corrispondenza solo dal fatto che **condividono la
stessa chiave**. È il difetto che vogliamo risolvere. Erano questi tre:

```python
richieste = {}   # host -> numero di richieste
errori = {}      # host -> numero di 5xx
latenze = {}     # host -> lista dei response time
```

Le informazioni su *web02*, ad esempio, non stavano in un posto solo: erano **sparse sui tre
dizionari**, e l'unico collante che le teneva insieme era la chiave `"web02"` ripetuta in tutti
e tre. Per ricostruire "lo stato di web02" si dovevano interrogare le tre strutture in parallelo:

```python
richieste["web02"]   # 200
errori["web02"]      # 19
latenze["web02"]     # [47, 1203, ...]
```

Il problema è che quell'allineamento era **a carico del programmatore**, non garantito dalla
struttura — ed è ciò che rende il pattern fragile: ogni host nuovo **va inizializzato a mano** in
**tutti e tre** i dizionari insieme, gli aggiornamenti vanno tenuti sincronizzati, e nulla nel
codice dichiara che quei tre dizionari descrivono lo stesso oggetto (è una convenzione che vive
solo nella testa di chi scrive).

### Il primo passo: dare ai dati un nome e una forma

Capovolgiamo la logica con le `dataclass`. **Una `dataclass` è una classe pensata per contenere
dati**: ne dichiariamo i campi e Python genera automaticamente il costruttore(il metodo di istanziazione), la stampa leggibile di una riga
e il confronto fra oggetti, cioè righe diverse (3 metodi essenziali)

---
**📌Nota sulla `dataclass`**

**Una `dataclass` è analoga ad un tracciato record**:è, in sostanza, la definizione della struttura di un record in memoria: dichiara quali campi compongono un record, in che ordine, e di che tipo. Proprio come un tracciato descrive il layout di un record — questi campi, in quest'ordine, di questo tipo — `Evento` (esempio di `dataclass`) dice "un evento è fatto di `host`, `status`, `rt_ms"`. È una descrizione di una forma dei dati,<br>
> e **ogni istanza (`Evento("web01", 200, 47)`) è un record conforme a quel tracciato**.

Il parallelo regge: definizione della struttura ↔ tracciato; istanza ↔ record concreto.

---
Definiamo tre `dataclass`, una per concetto:

- **`Evento`** — **una** richiesta letta dal log (host, status, response time).
- **`StatoHost`** — i **KPI aggregati** di un host. Qui `error_rate` non è un campo memorizzato
  ma una **proprietà**: viene *calcolato* dagli altri dati (errori ÷ richieste) ogni volta che
  serve. Non è un dato da tenere allineato, è una funzione dei dati.
- **`Soglie`** — raccoglie i **numeri magici** (`0.05`, `800`, `0.95`) in un unico oggetto
  **immutabile** (`frozen=True`): un solo posto dove leggerli e cambiarli.

Così, ad esempio, "lo stato di web02" diventa **un solo oggetto**, con i suoi campi sempre insieme e coerenti
per costruzione: niente più allineamento manuale.

In [ ]:
import math                              # funzioni matematiche di base; qui per math.ceil(), usato nel calcolo dell'indice del percentile (p95)
from collections import defaultdict      # dizionario con valore di default automatico; qui per raggruppare gli eventi per host senza inizializzare a mano ogni chiave
from dataclasses import dataclass        # la classe che genera in automatico __init__/__repr__/__eq__ dai campi; qui per definire i nostri record Evento, StatoHost, Soglie
from pathlib import Path                 # gestione moderna e portabile di file e cartelle; qui per leggere i file di log (glob, open, encoding) in modo pulito

@dataclass
class Evento:
    """Una singola richiesta letta da una riga di log."""
    host: str
    status: int
    rt_ms: int


@dataclass
class StatoHost:
    """KPI aggregati di un host."""
    host: str
    n_richieste: int
    n_errori: int
    p95_ms: int

    @property
    def error_rate(self) -> float:
        # derivato dai dati: niente da tenere "allineato a mano"
        return self.n_errori / self.n_richieste if self.n_richieste else 0.0


@dataclass(frozen=True)
class Soglie:
    """Le soglie di allarme, in un unico posto e immutabili."""
    error_rate_max: float = 0.05   # oltre questa quota di 5xx => critico
    p95_ms_max: int = 800          # oltre questa latenza p95 (ms) => critico
    percentile: float = 0.95       # quale percentile usare per la latenza

La cella precedente non produce output visibile, definisce soltanto le tre `dataclass`.

Una nota: i due package `math` e `defaultdict` non servono ancora qui — verranno usati nelle celle successive (calcolo del percentile e aggregazione), ma li importiamo già ora dato che questa è la cella che apre la parte idiomatica per le dataclass.

**Con quali criteri abbiamo definito i tre valori soglia?**

Questi tre valori hanno **statuti diversi**. Uno è una scelta matematica precisa, gli altri due sono **soglie operative convenzionali e didattiche** — plausibili, ma non calcolate da dati reali.

Vale la pena distinguerli didatticamente.

**`percentile = 0.95`** è il più "solido" dei tre valori soglia, ma è una **convenzione di settore**, non un valore tarato. La `p95` (insieme a `p99` e `p99.9`) è lo standard de facto nel monitoraggio delle prestazioni perché cattura "l'esperienza dei casi peggiori" <u>senza inseguire l'outlier estremo</u>: la `p95` dice "il 5% più lento sta sopra questo valore". Si usa la `p95` perché la media nasconde le code (come abbiamo visto con l'istogramma) e la `p100` (il massimo) è troppo ballerina, sensibile a un singolo evento. Quindi `0.95` riflette una pratica consolidata. La scelta tra `p95`, `p99` o `p99.9` dipende poi da quanto si è severi sui casi peggiori.

**`error_rate_max = 0.05`** (il 5%) è una **soglia di comodo, scelta per il modulo**. Non c'è un calcolo dietro: è semplicemente un valore *plausibile e tondo* che serve a far scattare l'allarme su `web02` (che genera circa il 9-12% di errori) e a lasciar tranquilli gli host sani (circa l'1-2%). In un contesto reale, un error rate "accettabile" del 5% sarebbe spesso considerato *altissimo*: i target di affidabilità seri si misurano in "noven" — 99,9% di richieste riuscite significa un budget d'errore dello 0,1%, non del 5%. Qui il 5% è **tarato sui dati *simulati* del notebook, non su uno standard di produzione**.

**`p95_ms_max = 800`** (800 ms) è anch'esso un **valore didattico plausibile**, non derivato. 800 ms è una latenza "percepibile come lenta" per una richiesta web — un numero ragionevole da mettere in un esempio — e separa nettamente gli host sani del notebook (p95 circa 90-110 ms) da `web02` (circa 2000 ms). Ma la soglia "giusta" dipende interamente dal contesto: 800 ms è inaccettabile per un'API interna che deve rispondere in decine di ms, e ottimo per un'operazione pesante che ne può richiedere parecchi.

Nota **metodologica**: una soglia sensata non si inventa, si **deriva**. In produzione la si fissa a partire da (a) un **obiettivo di servizio** — uno SLA, es. "il 99% delle richieste sotto 500 ms" — concordato con chi usa il servizio; oppure (b) la **baseline storica** del sistema, osservando come si comporta normalmente e fissando la soglia in termini di scostamento dal comportamento abituale.

👉 Ecco perchè queste soglie le abbiamo raccolte nella `dataclass Soglie`: proprio perché sono **parametri di configurazione** — valori che cambieranno da ambiente ad ambiente e che andranno tarati sul sistema reale — ha senso tenerli in un unico posto, con un nome, separati dalla logica (e non dentro il codice). **Sono decisioni, non costanti universali**.

**`@` è un decoratore.**

Per capire la riga teniamo separati **due attori distinti**: la funzione `dataclass`, che
fa il lavoro vero, e il simbolo `@`, che serve solo ad applicarla.

**Cosa fa `dataclass` (la funzione).** In Python una classe è essa stessa un oggetto, e come
tale possiamo passarla a una funzione. `dataclass` è esattamente questo: una funzione che
**riceve una classe, ne legge i campi dichiarati e le aggiunge i metodi standard** che
altrimenti dovremmo scrivere a mano. Dai campi `host`, `status`, `rt_ms` genera per noi:

- `__init__`, il costruttore → possiamo fare `Evento("web01", 200, 47)`;
- `__repr__`, la stampa leggibile → stampando l'oggetto vediamo `Evento(host='web01', status=200, rt_ms=47)`;
- `__eq__`, il confronto → due `Evento` con gli stessi valori risultano uguali.

È **`dataclass` a produrre questo potenziamento.** Restituisce la stessa classe, arricchita di
quei metodi.

📌 `dataclass` è una funzione — più esattamente, una funzione decoratrice. Non è una classe, nonostante il nome contenga la parola "class".

Il punto è distinguere `dataclass` (lo strumento) da "una dataclass" (il risultato). Sono due cose diverse, e il linguaggio comune le confonde:
* dataclass — l'oggetto che importiamo da dataclasses — è una funzione. La chiamiamo (o la applichiamo con @) passandole una classe.
* "una dataclass" — ciò che otteniamo — è una classe: Evento, StatoHost, Soglie sono classi a tutti gli effetti.

Quindi la funzione si chiama `dataclass` perché il suo scopo è produrre una classe-per-dati: il nome descrive cosa fabbrica, non cosa è. Un po' come una "macchina per il pane" è una macchina, non pane: `dataclass` è una funzione che trasforma classi, non una classe.

**Cosa fa `@` (il decoratore).** Il simbolo `@`, scritto sulla riga subito prima di una
definizione, è semplicemente una **scorciatoia per applicare** quella funzione alla classe.

Scrivere:

```python
  @dataclass
  class Evento:
      ...
```

equivale esattamente a definire la classe e poi scrivere `Evento = dataclass(Evento)`. La `@`
non aggiunge nulla di suo: prende la classe definita qui sotto, la passa a `dataclass` e rimette
il risultato sotto lo stesso nome. È solo zucchero sintattico che ci risparmia quella riga.

📌 **In sintesi:** il lavoro lo fa `dataclass`, la `@` lo aziona in modo pulito. Senza tutto questo
dovremmo scrivere `__init__`, `__repr__` e `__eq__` a mano; con `@dataclass`, dalla sola
dichiarazione dei campi, li otteniamo gratis.

Le `dataclass` sono OOP?

Domanda acuta, e la risposta onesta è: **in parte sì, ma non nel senso pieno del termine.** Stiamo usando *classi*, quindi tecnicamente è codice orientato agli oggetti — ma le `dataclass` qui sono usate come **strutture dati**, non come oggetti "ricchi" nel senso classico dell'OOP.

La distinzione è importante e vale la pena chiarirla, perché tocca proprio il filo conduttore del corso.

L'OOP "classica" unisce in un oggetto **dati + comportamento**: l'oggetto custodisce il suo stato e si fa carico delle operazioni su quello stato, spesso mutandolo (`conto.preleva(50)` modifica il saldo dentro l'oggetto). Le nostre `dataclass`, invece, sono volutamente quasi solo **dati**: `Evento` è un contenitore di tre valori, `Soglie` è addirittura immutabile (`frozen=True`). Non "fanno" cose: vengono *passate* alle funzioni che fanno le cose. È più vicino a un `record` o a uno `struct` che a un oggetto pieno.

Ed è una scelta deliberata, in tensione con uno dei principi cardine dell'OOP. L'incapsulamento classico dice "metti il comportamento *dentro* l'oggetto, insieme ai dati". Noi facciamo il contrario: teniamo i **dati** nelle `dataclass` e la **logica** in funzioni separate (`parse_riga`, `aggrega`, `percentile`). È esattamente il principio "separare la logica dall'I/O, la logica in funzioni pure" che guida tutto il modulo. Se mettessimo il calcolo *dentro* gli oggetti, rinunceremmo in parte a quella separazione netta che rende le funzioni pure così facili da testare.

C'è un'unica eccezione, e illumina la differenza: la **proprietà** `error_rate` su `StatoHost`. Quello *è* comportamento attaccato ai dati — un piccolo pezzo di logica che vive nell'oggetto. Ma è un caso limite scelto con cura: è un valore *derivato* dagli altri campi (errori ÷ richieste), banale e senza effetti collaterali, quindi metterlo lì non intacca la testabilità. Una proprietà calcolata è un compromesso elegante, non una deriva verso oggetti che fanno tutto.

Per inquadrarlo in una frase: stiamo usando le classi per **dare forma ai dati**, non per costruire una gerarchia di oggetti che incapsulano comportamento. Niente ereditarietà, niente metodi che mutano lo stato, niente polimorfismo. **È uno stile spesso chiamato *data-oriented* — moderno e molto comune in Python per il codice di analisi e infrastruttura** — che prende dall'OOP la parte utile qui (un tipo con un nome e dei campi tipizzati) lasciando da parte il resto.


📌 **Approccio di questo notebook**

Quello che abbiamo adottato si chiama stile **data-oriented** (orientato ai dati): teniamo i dati in <u>strutture passive</u> — le dataclass `Evento`, `StatoHost`, `Soglie`, che sono essenzialmente **record** — e la <u>logica</u> in funzioni separate (`parse_riga`, *aggrega*, *percentile*) che ricevono quei dati ed elaborano. Dati di qua, comportamento di là. È diverso dall'OOP "piena", dove dati e comportamento stanno insieme dentro l'oggetto, che si fa carico delle operazioni sul proprio stato.

Il notebook mescola due paradigmi, e lo fa deliberatamente.

C'è un po' di OOP, nel senso che usiamo le classi per dare forma ai dati (e la proprietà *error_rate* è comportamento attaccato a un oggetto). Ma sono **classi usate come contenitori**, non come oggetti "ricchi".

C'è soprattutto uno stile procedurale/funzionale orientato ai dati: il cuore del lavoro lo fanno funzioni pure che trasformano strutture dati immutabili o quasi.

Il motivo per cui abbiamo scelto questo stile, ed è il filo che tiene insieme tutto il modulo, è la **testabilità**. Tenendo la logica in funzioni pure, separate dai dati e dall'I/O, possiamo verificarla in isolamento — è esattamente ciò che ci ha permesso di scrivere (più avanti) `test_parse_riga_valida` e `test_aggrega_kpi` con un semplice input→output.

> Se avessimo incapsulato il calcolo dentro oggetti che mutano il proprio stato e leggono file, **quei test sarebbero stati molto più difficili**. La scelta paradigmatica, in altre parole, non deve essere ideologica: discende dall'obiettivo "logica pura e testabile" che guida l'intera trasformazione da script a strumento.

In sintesi: non abbiamo abbracciato l'OOP in senso stretto, abbiamo usato **le classi come strumento per modellare i dati dentro uno stile prevalentemente data-oriented e funzionale** — perché è quello che rende il codice di infrastruttura più semplice da leggere, testare e mantenere. L'OOP piena (ereditarietà, polimorfismo) tornerà utile quando servirà davvero, ad esempio per dare un'interfaccia comune a sorgenti diverse come Oracle, SQL Server e Splunk.

### I type hints

Accanto ai nomi compaiono i **type hints**: annotazioni che dichiarano *di che tipo* è un dato o
cosa restituisce una funzione.

```python
host: str                          # il parametro "host è una stringa"
def error_rate(self) -> float:     # "questa funzione restituisce un float"
```

Un'annotazione (*hint*) può anche essere composta: `int | None` significa "un intero **oppure** `None`".

Il punto che spiazza chi arriva dallo scripting: **Python non li controlla a runtime**. Sono
annotazioni *opzionali*; il codice gira esattamente uguale con o senza. Se una funzione annotata
`def f(x: int)` viene chiamata con `f("ciao")`, Python non solleva alcun errore solo perché il
tipo non combacia — resta un linguaggio a tipizzazione dinamica. I type hint **non** trasformano
Python in un linguaggio a tipi statici: aggiungono un livello di informazione sopra il codice,
senza cambiarne l'esecuzione.

A cosa servono, allora, se non vengono verificati? A due cose:

1. **Documentazione per chi legge (per chiarezza).** La firma `def aggrega(eventi: list[Evento], soglie: Soglie)
   -> list[StatoHost]` dice da sola cosa entra e cosa esce, senza dover leggere il corpo della
   funzione. Ed è documentazione che non può "andare fuori sincrono" col codice come un commento,
   perché è parte della firma stessa.
2. **Strumenti di analisi statica.** L'editor li usa per l'autocompletamento e per segnalare in
   tempo reale gli usi incoerenti; un *type checker* come `mypy` (o `pyright`) analizza tutto il
   codice **senza eseguirlo** e trova gli errori di tipo prima che diventino bug a runtime —
   passare una stringa dove serve un numero, dimenticare che una funzione può restituire `None`,
   e così via. Un controllo di qualità in fase di sviluppo, non in produzione di notte.

👉 **La firma non è vincolante per l'interprete (a runtime resta dinamico), ma è vincolante per gli strumenti di analisi**:<br>
> ed è proprio questo che la rende documentazione affidabile invece che una semplice promessa scritta a margine.

Il "vincolo", insomma, c'è, ma scatta in fase di sviluppo (mypy, editor), non durante l'esecuzione.

Un'ultima precisazione, perché qui i type hint hanno **due ruoli diversi**. Di solito sono
*opzionali* (utili agli strumenti). Ma **dentro una `dataclass` diventano strutturali**: scrivere
`host: str` è ciò che *definisce* il campo `host` — senza l'annotazione di tipo, la `dataclass`
non saprebbe nemmeno che quel campo esiste. Quindi in `Evento`, `StatoHost` e `Soglie` non sono
un di più: sono il modo in cui dichiariamo i campi.

---
📌 **Non abbiamo creato un modello dati nuovo**, abbiamo dato un nome a un modello dati che era già lì, **implicito**. Lo script *as-is* aveva il concetto di "evento" e di "stato di un host" — solo che non li chiamava per nome e li teneva in variabili sciolte e dizionari paralleli. Le `dataclass` non aggiungono concetti: rendono visibile e robusta la struttura che il codice grezzo già sottintendeva.

Come li chiamava l'*as-is*? **Non li chiamava affatto** — ed è proprio questo il punto. Non aveva nomi *alternativi* per "evento" e "stato di un host"; quei concetti esistevano solo come **dati sparsi**, mai raccolti sotto un'etichetta unica. Vediamo i due casi separatamente, perché si manifestavano in modo diverso.

**Il concetto di "evento"** nell'*as-is* non aveva nome perché non esisteva nemmeno come *oggetto*: viveva solo come **tre variabili temporanee** dentro il ciclo di parsing, che apparivano e sparivano a ogni riga:

```python
  for token in parti[1:]:
      chiave, valore = token.split("=")
      if chiave == "host":   host = valore
      elif chiave == "status": status = int(valore)
      elif chiave == "rt_ms":  rt = int(valore)
  # qui esistono host, status, rt -- tre variabili sciolte, non un "evento"
```

Le informazioni di una richiesta c'erano tutte (`host`, `status`, `rt`), ma **slegate fra loro**: niente diceva "queste tre cose insieme *sono* un evento". Venivano usate subito (per aggiornare i contatori) e poi sovrascritte alla riga successiva. Quindi l'*as-is* non chiamava l'evento con un altro nome: semplicemente non lo *reificava*, non ne faceva *una cosa con un'identità*. È esattamente ciò che `Evento` rende esplicito — le stesse tre variabili, ora raccolte in **un oggetto che *si chiama* evento**.

**Il concetto di "stato di un host"**, invece, un nome ce l'aveva — ma **frammentato in tre**. Non si chiamava "stato di web02": si chiamava `richieste["web02"]`, `errori["web02"]`, `latenze["web02"]`.

```python
  richieste = {}   # \
  errori = {}      #  > lo "stato di un host" sparso su tre nomi diversi
  latenze = {}     # /
```

Lo stato di un host esisteva, ma **non aveva un nome unico**: era distribuito su tre dizionari, e per ricostruirlo bisognava interrogarli tutti e tre con la stessa chiave. Il "nome" del concetto, di fatto, era la **chiave** `"web02"` ripetuta — l'unico collante che diceva "questi tre pezzi appartengono allo stesso host". `StatoHost` prende quei tre pezzi e dà loro un nome solo.

> Quindi, l'*as-is* chiamava l'"evento" con **nessun nome** (tre variabili usa-e-getta) e lo "stato di un host" con **tre nomi paralleli** invece di uno.

In entrambi i casi i concetti c'erano — il codice li *sottintendeva*, perché manipolava esattamente quei dati — ma non erano mai stati promossi a "cose con un nome". È la differenza tra *avere* implicitamente una struttura e *dichiararla* esplicitamente:<br>
> le `dataclass` non introducono i concetti, danno loro un'identità e un'etichetta che prima vivevano solo nella logica sparsa del codice.

C'è una formula che rende bene l'idea: nell'as-is quei concetti erano **"nella testa del programmatore", non nel codice**. Chi aveva scritto lo script *sapeva* che `host`, `status`, `rt` formavano un evento, e che i tre dizionari descrivevano lo stesso host — ma quella conoscenza non era scritta da nessuna parte, viveva solo nell'intenzione. Le `dataclass` la trasferiscono dalla testa del programmatore al codice.

---

Per maggior chiarezza vediamo **due esempi di `dataclass` lontani dal mondo del monitoraggio**, scelti per mostrare due tipi di utilità diversi.

**Esempio 1 — Un punto geometrico**

Il caso "da manuale", utile per vedere il valore aggiunto in poche righe:

```python
  from dataclasses import dataclass

  @dataclass
  class Punto:
      x: float
      y: float

      def distanza_da(self, altro: "Punto") -> float:
          return ((self.x - altro.x) ** 2 + (self.y - altro.y) ** 2) ** 0.5
```

L'utilità qui è la **comodità immediata** che il decoratore ci regala. Con quattro righe abbiamo già:

```python
  a = Punto(1, 2)
  b = Punto(4, 6)

  print(a)                 # Punto(x=1, y=2)   -> stampa leggibile (__repr__)
  print(a == Punto(1, 2))  # True              -> confronto per valore (__eq__)
  print(a.distanza_da(b))  # 5.0               -> il nostro metodo
```

Senza `@dataclass` dovremmo scrivere a mano `__init__`, `__repr__` e `__eq__` solo per arrivare a questo. Si nota anche un dettaglio: una `dataclass` **può avere metodi propri** (`distanza_da`) — non è obbligata a essere solo dati. Resta comodo persino quando un po' di comportamento ce lo mettiamo.

**Esempio 2 — La configurazione di un'applicazione**

Il caso più "professionale", che mostra l'utilità su un altro fronte:

```python
  from dataclasses import dataclass

  @dataclass(frozen=True)
  class ConfigDB:
      host: str
      porta: int = 5432              # valore predefinito
      nome_db: str = "produzione"
      timeout_s: float = 30.0
      ssl: bool = True
```

```python
  cfg = ConfigDB(host="db.interno.local", porta=1521)
  print(cfg.timeout_s)     # 30.0  -> il default, non l'abbiamo specificato
  # cfg.porta = 1522       -> ERRORE: frozen=True la rende immutabile
```

Qui l'utilità è di tipo diverso, e tocca tre punti pratici:

- **Un contenitore unico e tipato** al posto di una manciata di variabili sciolte o di un dizionario `config["porta"]`. Si accede con `cfg.porta` (l'editor autocompleta, un refuso come `cfg.prota` viene segnalato), invece di `config["prota"]` che fallirebbe solo a runtime.
- **Valori predefiniti** dichiarati una volta, in chiaro: chi legge la classe vede subito cosa è obbligatorio (`host`) e cosa ha un default.
- **`frozen=True`** per l'immutabilità: una configurazione non deve cambiare mentre il programma gira, e renderla immutabile previene modifiche accidentali — esattamente la stessa ragione per cui nel notebook abbiamo congelato `Soglie`.

**I due tipi di utilità, in sintesi**

I due esempi illustrano i due motivi principali per cui si usano le `dataclass`:

1. **Risparmiare codice ripetitivo** (il `Punto`): otteniamo costruttore, stampa e confronto gratis, e possiamo aggiungere metodi quando servono.
2. **Dare forma, nome e sicurezza ai dati** (la `ConfigDB`): un tipo esplicito con campi tipizzati, default e immutabilità, al posto di dizionari o variabili sparse — più leggibile e meno soggetto a errori.

Un terzo uso frequentissimo, che unisce i due, è rappresentare i **record** che entrano ed escono da un programma: una riga di un CSV, un elemento di una risposta JSON, un messaggio in coda. È esattamente il ruolo di `Evento` nel notebook — e per quello, quando i dati arrivano da fonti esterne e vogliamo anche *validarli* (non solo dichiararne i tipi), il passo successivo naturale è una libreria come **Pydantic**, che useremo più avanti.

### Il parsing come funzione pura

Isoliamo la trasformazione *"riga di testo → `Evento`"* in una funzione **pura**: nessun I/O,
nessun logging, nessuno stato globale. Restituisce un `Evento` se la riga è valida, `None` se
non lo è. Essendo pura, è **banale da testare** (lo faremo) e **difensiva** per costruzione —
non assume nulla sull'input:

- `partition("=")` invece di `split("=")`: non esplode se manca o c'è più di un `=`;
- controlla che **tutti** i campi attesi siano presenti;
- intercetta i valori **non numerici** con `try/except`.

In [ ]:
def parse_riga(riga: str) -> Evento | None:
    """Trasforma una riga di log in un Evento, o restituisce None se la riga non interpretabile.

    Funzione PURA: nessun I/O, nessun logging. Tutta la fragilità del mondo reale
    (campi mancanti, valori non numerici, righe spazzatura) è gestita qui, in un
    punto solo, ben definito e testabile.
    """
    # 'campi' è un dizionario di comodo, di passaggio (per non sporcare gli altri dati)
    campi: dict[str, str] = {}              # inizializzazione
    for token in riga.split()[1:]:          # ciclo sui token (gli elementi base) della riga
                                            # salta il token 0 (il timestamp): riga.split()[0]
        if "=" not in token:
            return None                     # token che non è una coppia chiave=valore
        chiave, _, valore = token.partition("=")   # robusto: niente unpack che esplode
        campi[chiave] = valore

    # devono esserci TUTTI e tre i campi, altrimenti la riga non è utilizzabile
    if not {"host", "status", "rt_ms"}.issubset(campi):
        return None

    try:
        return Evento(
            host=campi["host"],
            status=int(campi["status"]),
            rt_ms=int(campi["rt_ms"]),
        )
    except ValueError:
        return None                          # status mancante o rt_ms non numerici


### L'I/O ai bordi: lettura robusta

Ora la lettura dei file — l'**I/O**, confinato in una funzione sua. Qui concentriamo tutto ciò
che riguarda *come* arrivano i dati, lasciando il resto del programma indifferente alla loro
provenienza:

- `with ... open(...)` (**context manager**): il file si chiude da solo, anche in caso di errore;
- **encoding esplicito**: niente sorprese tra ambienti;
- `cartella.glob("*.log")`: leggiamo **solo** i file di log — il `README.txt` estraneo è ignorato
  per costruzione, non per fortuna;
- iteriamo il file **riga per riga** (è un generatore): un log da gigabyte non viene caricato
  tutto in memoria — cruciale sui log ruotati delle vere infrastrutture in produzione;
- le righe invalide vengono **contate e segnalate** con `logging`, non fanno saltare nulla.

In [ ]:
def leggi_eventi(cartella: Path) -> list[Evento]:
    """Legge tutti i file *.log della cartella e restituisce gli Eventi validi.

    L'I/O sta tutto qui. Le righe non interpretabili vengono contate e segnalate
    via logging, ma NON interrompono l'elaborazione.
    """
    eventi: list[Evento] = []
    scartate = 0

    for percorso in sorted(cartella.glob("*.log")):      # solo i .log (ordinati): il README viene ignorato
        with percorso.open(encoding="utf-8") as f:       # context manager + encoding esplicito
            for numero, riga in enumerate(f, start=1):   # streaming: una riga alla volta
                riga = riga.strip()
                if not riga:
                    continue                              # salta le righe vuote
                evento = parse_riga(riga)                 # la logica pura di parsing (la separazione dei concern)
                if evento is None:
                    scartate += 1
                    logger.warning("%s:%d riga non valida, saltata: %r",
                                   percorso.name, numero, riga)
                    continue
                eventi.append(evento)

    if scartate:
        logger.info("Righe scartate in totale: %d", scartate)
    logger.info("Eventi validi letti: %d", len(eventi))
    return eventi


### Il calcolo come funzione pura: aggregazione e KPI

Il cuore dell'analisi, di nuovo come funzioni **pure** — ricevono dati, restituiscono dati,
non leggono né stampano nulla:

- `percentile(...)` calcola il p-esimo percentile con il metodo *nearest-rank* (esplicito e
  prevedibile), al posto dell'indice calcolato "al volo" della versione *as-is*;
- `aggrega(...)` raggruppa gli eventi per host (con un `defaultdict`, niente inizializzazione a
  mano) e produce gli `StatoHost`;
- `diagnosi(...)` decide se un host è critico e **perché**, restituendo l'elenco dei motivi
  (lista vuota = host sano).

Separare il *calcolo* dal *giudizio* (`diagnosi`) dal *report* significa che ognuno si può
cambiare e testare per conto suo.

**Due note di chiarimento**:

**Cos'è il *nearest-rank***

Il **nearest-rank** è un algoritmo per calcolare un percentile scegliendo, fra i valori realmente presenti nei dati, **quello in una posizione precisa** della lista ordinata — senza interpolare, senza inventare valori intermedi. Il p95 "nearest-rank" è letteralmente *uno dei response time osservati*, non una media fra due.

La definizione operativa, in tre passi:
1. **Ordina** i valori dal più piccolo al più grande.
2. **Calcola il rango** (la posizione) con la formula `rango = ⌈p × n⌉` — il *ceiling* (arrotondamento per eccesso) di "percentile × numero di valori".
3. **Prendi il valore** in quella posizione.

È esattamente ciò che fa la funzione nel notebook:

```python
def percentile(valori, frazione):
    ordinati = sorted(valori)
    indice = max(0, math.ceil(frazione * len(ordinati)) - 1)  # -1: gli indici partono da 0
    return ordinati[indice]
```

Un esempio concreto chiarisce tutto. Supponiamo 10 latenze ordinate e di volere il p95:

```
posizione:  1    2    3    4    5    6    7    8    9    10
valore:     40   45   50   55   60   70   80   90   120  900
```

`rango = ⌈0.95 × 10⌉ = ⌈9.5⌉ = 10` → prendiamo il **10° valore = 900**. Il p95 nearest-rank è 900: il primo valore tale che almeno il 95% delle osservazioni gli sta sotto o pari.

Il `-1` nel codice serve solo perché le liste Python sono indicizzate da 0: il "10° valore" sta all'indice 9. E il `max(0, …)` protegge il caso di una lista con un solo elemento.

**Perché lo preferiamo all'as-is.** Lo script as-is faceva `ordinate[int(len * 0.95)]`: `int(...)` **tronca** invece di arrotondare per eccesso, dando un indice leggermente diverso e con una logica meno difendibile (con 10 elementi: `int(9.5) = 9`, l'indice 9 → 10° valore — qui coincide, ma su altri conteggi i due metodi divergono). Il punto non è solo l'eventuale differenza numerica: è che il nearest-rank è un metodo **con un nome e una definizione standard**, mentre l'altro era un calcolo improvvisato "al volo".

Una precisazione onesta: il nearest-rank è **uno** degli algoritmi per i percentili, non l'unico. `numpy.percentile`, di default, usa l'**interpolazione lineare** — se il rango cade "fra" due posizioni, restituisce una media pesata dei due valori, producendo un numero che *non* è necessariamente fra quelli osservati. Per il monitoraggio il nearest-rank va benissimo ed è anche più intuitivo ("il p95 è una latenza che è davvero successa"), ma è bene sapere che esistono convenzioni diverse, perché due strumenti possono dare p95 leggermente diversi sugli stessi dati proprio per questo.

**Come si decide se un host è critico e perché?**
Sempre per il principio della separazione dei concern, questa logica deve essere incapsulata in una funzione.

Il "perché" è la parte interessante: la seguente funzone `diagnosi` non restituisce un semplice sì/no, ma **l'elenco dei motivi** per cui un host è critico. Ecco la funzione:

```python
  def diagnosi(stato: StatoHost, soglie: Soglie) -> list[str]:
      motivi = []
      if stato.error_rate > soglie.error_rate_max:
          motivi.append(f"error_rate {stato.error_rate:.1%} > {soglie.error_rate_max:.1%}")
      if stato.p95_ms > soglie.p95_ms_max:
          motivi.append(f"p95 {stato.p95_ms}ms > {soglie.p95_ms_max}ms")
      return motivi
```

Il meccanismo, passo per passo:

- Confronta **ogni KPI dell'host con la soglia corrispondente**: l'`error_rate` contro `error_rate_max` (0.05), la `p95_ms` contro `p95_ms_max` (800).
- Per **ogni** soglia superata, aggiunge alla lista `motivi` una **stringa che descrive il problema** — non solo "c'è un problema", ma quale, con i numeri: `"error_rate 9.5% > 5.0%"`.
- Restituisce la **lista dei motivi**.

La chiave è come questa lista codifica il verdetto, ed è una scelta di design elegante:

- **lista vuota `[]`** → nessuna soglia superata → **host sano**;
- **lista non vuota** → host **critico**, e il contenuto della lista dice *esattamente perché* (può contenere uno o due motivi, se sfora entrambe le soglie).

Quindi il "sì/no" e il "perché" sono **la stessa cosa**: la presenza di motivi *è* la criticità, e i motivi stessi sono la spiegazione. Chi chiama la funzione usa la lista in due modi con una sola informazione — `if diagnosi(stato, soglie):` per sapere *se* è critico (una lista vuota è "falsa" in Python, una piena è "vera"), e scorre i motivi per *stamparli* nel report. È esattamente ciò che fa `stampa_report`:

```python
  motivi = diagnosi(s, soglie)
  esito = "OK" if not motivi else "CRITICO"
  ...
  for m in motivi:
      print(f"         -> {m}")
```

Due pregi da sottolineare. Primo: `diagnosi` è una **funzione pura** che riceve uno `StatoHost` e le `Soglie` e restituisce dati (la lista) — non stampa nulla, non decide *come* mostrare il verdetto (della diagnosi). Separa il **giudizio** ("è critico e perché") dalla **presentazione** ("come lo scrivo nel report"), così ognuno si può cambiare e testare per conto suo. Secondo: restituire i *motivi* invece di un booleano rende lo strumento **diagnostico**, non solo un allarme — dice al sistemista non soltanto "guarda web02", ma "guarda web02 *perché* l'error rate è al 9,5% e la p95 a 2040 ms", che è l'informazione con cui parte davvero il troubleshooting.

In [ ]:
def percentile(valori: list[int], frazione: float) -> int:
    """p-esimo percentile (metodo nearest-rank) su una lista NON vuota."""
    ordinati = sorted(valori)                                 # ordinamento
    indice = max(0, math.ceil(frazione * len(ordinati)) - 1)  # calcolo percentile
    return ordinati[indice]                                   # 'indice' è la posizione del percentile nella lista valori ordinata


def aggrega(eventi: list[Evento], soglie: Soglie) -> list[StatoHost]:
    """Raggruppa gli eventi per host e calcola i KPI. Funzione PURA."""
    per_host: dict[str, list[Evento]] = defaultdict(list)   # niente init "a mano"
    for e in eventi:
        per_host[e.host].append(e)

    stati: list[StatoHost] = []
    for host, lista in sorted(per_host.items()):
        n_richieste = len(lista)
        n_errori = sum(1 for e in lista if e.status >= 500)
        p95 = percentile([e.rt_ms for e in lista], soglie.percentile)
        stati.append(StatoHost(host=host, n_richieste=n_richieste,
                               n_errori=n_errori, p95_ms=p95))
    return stati


def diagnosi(stato: StatoHost, soglie: Soglie) -> list[str]:
    """Elenca i motivi per cui un host è critico. Lista vuota = host sano. PURA."""
    motivi = []
    if stato.error_rate > soglie.error_rate_max:
        motivi.append(f"error_rate {stato.error_rate:.1%} > {soglie.error_rate_max:.1%}")
    if stato.p95_ms > soglie.p95_ms_max:
        motivi.append(f"p95 {stato.p95_ms}ms > {soglie.p95_ms_max}ms")
    return motivi


### Il report (I/O) e l'orchestrazione

Definiamo **l'ultima funzione**.

Il report stampa a video: è **I/O**, quindi sta in una funzione sua, separata da tutti i concern precedenti.


In [ ]:
def stampa_report(stati: list[StatoHost], soglie: Soglie) -> None:
    """Stampa il report a video. È I/O: nessun calcolo qui dentro."""
    print("=== Report host ===")
    for s in stati:
        motivi = diagnosi(s, soglie)
        esito = "OK" if not motivi else "CRITICO"
        print(f"{s.host:<8} | rich: {s.n_richieste:>4} | "
              f"err: {s.error_rate:>6.1%} | p95: {s.p95_ms:>5}ms | {esito}")
        for m in motivi:
            print(f"         -> {m}")

### L'orchestratore

La funzione `main` è l'**orchestratore**: mette in fila i passi (leggi → aggrega → riporta) senza contenere
logica di dettaglio. Si legge come un indice del programma.

In [ ]:
def main(cartella: Path, soglie: Soglie = Soglie()) -> list[StatoHost]:
    """Orchestratore: legge, aggrega, riporta. Restituisce gli stati (utile per i test)."""
    eventi = leggi_eventi(cartella)        # I/O ai bordi
    stati = aggrega(eventi, soglie)        # logica pura
    stampa_report(stati, soglie)           # I/O ai bordi
    return stati


Eseguiamo lo strumento sui **log reali** (`logs_reali/`, quelli con le righe sporche e il
file estraneo). Le righe invalide compaiono come `WARNING` di `logging` — **separate** dal
report — e l'elaborazione arriva in fondo. Dove l'as-is si interrompeva, lo strumento
**sopravvive e ti dice cosa ha scartato**.

In [ ]:
stati = main(Path("logs_reali"))


E' **esattamente** il risultato atteso: lo strumento gira sui dati "reali" (`logs_reali/`), sopravvive alle righe sporche e produce il report corretto.

👉 È la prova che la versione idiomatica fa ciò che l'as-is non riusciva a fare. Commentiamo l'output nelle sue due parti.

**Parte 1 — La diagnostica (`logging`)**

Le prime righe sono i messaggi di `logging`, **separati dall'output del report** — esattamente lo scopo per cui usiamo `logging` invece di `print`:

```text
WARNING web02.log:201 riga non valida, saltata: '...host=web02 status=500'      (manca rt_ms)
WARNING web02.log:202 riga non valida, saltata: '...status=200 rt_ms=NaN'        (non numerico)
WARNING web02.log:203 riga non valida, saltata: "QUESTA RIGA E' ROTTA"           (niente chiave=valore)
WARNING web02.log:204 riga non valida, saltata: '...status=200 rt_ms=55'         (manca host)
INFO Righe scartate in totale: 4
INFO Eventi validi letti: 600
```

Qui si chiude il cerchio con la cella delle fragilità: sono **le stesse quattro righe sporche** che, date al parser *as-is*, lo facevano crashare o sbagliare in silenzio. Ora `parse_riga` le riconosce come invalide, `leggi_eventi` le **salta e le segnala** (con tanto di file e numero di riga, `web02.log:201` — utilissimo per ritrovarle), e l'elaborazione **arriva in fondo**. Lo strumento non solo sopravvive: ti dice *cosa* ha scartato e *dove*.

Un paio di numeri che tornano: **600 eventi validi** = 3 host × 200 righe pulite ciascuno; le 4 righe sporche erano *in più*, aggiunte solo a `web02.log` (righe 201-204), e infatti vengono tutte scartate. Quindi 604 righe lette, 4 scartate, 600 valide. I conti chiudono.

**Parte 2 — Il report**

```text
web01 | rich: 200 | err: 0.0% | p95:   88ms | OK
web02 | rich: 200 | err: 9.5% | p95: 1966ms | CRITICO
        -> error_rate 9.5% > 5.0%
        -> p95 1966ms > 800ms
web03 | rich: 200 | err: 1.0% | p95:  109ms | OK
```

Il verdetto è coerente con tutto ciò che abbiamo costruito:

- **`web02` è `CRITICO`**, e `diagnosi` mostra **entrambi i motivi**: `error_rate 9.5% > 5.0%` e `p95 1966ms > 800ms`. Sono superate tutte e due le soglie, quindi la lista dei motivi contiene due voci — ed è proprio il comportamento "decide se è critico *e perché*" di cui parlavamo. Non dice solo "guarda web02", dice *perché*.
- **`web01` e `web03` sono `OK`**: error rate 0,0% e 1,0%, p95 88 e 109 ms, ben sotto entrambe le soglie. Nessun motivo → lista vuota → host sano.

C'è un dettaglio che vale la pena cogliere, e che conferma la robustezza: `web02` ha **`rich: 200`**, non 204. Le 4 righe sporche erano state aggiunte a `web02.log`, ma essendo invalide vengono scartate *prima* dell'aggregazione — quindi `web02` viene giudicato sui suoi 200 eventi **validi**. Le righe rotte non inquinano i KPI: né gonfiano il conteggio delle richieste, né alterano error rate o p95. È la differenza tra uno strumento che scarta in modo pulito e uno che lascerebbe passare dati monchi.

**Il confronto che chiude il discorso**

Vale la pena affiancarlo mentalmente all'output della versione as-is, perché il numeri di `web02` sono leggermente diversi: as-is dava `p95: 2040ms`, qui `p95: 1966ms`. **Non è un errore** — è la differenza tra i due metodi di calcolo del percentile di cui parlavamo: l'as-is usava `int(len * 0.95)` (troncamento), la versione idiomatica usa il *nearest-rank* con `math.ceil`. Indici di poco diversi, stesso ordine di grandezza, stessa conclusione (p95 ben oltre 800 ms). L'error rate (9,5%) invece coincide, perché lì il calcolo è identico.

👉 In sintesi: l'output è corretto e racconta la storia del modulo in tre righe di log e quattro di report — lo strumento legge dati reali e sporchi, **scarta e segnala** ciò che non va, **non si interrompe**, e produce un verdetto **diagnostico** (critico *e* perché) su dati puliti dalle imperfezioni. È esattamente ciò che l'as-is non sapeva fare.

### Da funzione a strumento: CLI, *exit code*, punto di ingresso

L'ultimo passo che trasforma il codice in uno **strumento da terminale**:

- i percorsi e le soglie diventano **parametri** via `argparse` — niente più valori cablati;
- un **exit code** (0 = tutto OK, 1 = almeno un host critico) permette a cron/scheduler di
  capire l'esito *senza leggere l'output*: è così che un tool si integra in una pipeline;
- il blocco `if __name__ == "__main__"` rende il file **eseguibile** da riga di comando **e**
  **importabile** come modulo (per i test, o per riusarne le funzioni altrove).

In uno script reale lo lancieremmo così:

```bash
python monitor.py --cartella /var/log/app --error-rate-max 0.03
echo $?   # 0 oppure 1
```

Qui nel notebook simuliamo la riga di comando passando una lista esplicita a `parse_args`.

In [ ]:
import argparse
import sys


def parse_args(argv=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Analizza i log applicativi e segnala gli host critici.")
    parser.add_argument("--cartella", type=Path, default=Path("logs_reali"),
                        help="Cartella contenente i file *.log")
    parser.add_argument("--error-rate-max", type=float, default=0.05,
                        help="Soglia di error rate oltre la quale l'host è critico")
    parser.add_argument("--p95-ms-max", type=int, default=800,
                        help="Soglia di latenza p95 (ms) oltre la quale l'host è critico")
    return parser.parse_args(argv)


def cli(argv=None) -> int:
    """Punto di ingresso: traduce gli argomenti, esegue, restituisce l'exit code."""
    args = parse_args(argv)
    soglie = Soglie(error_rate_max=args.error_rate_max, p95_ms_max=args.p95_ms_max)
    stati = main(args.cartella, soglie)
    critici = [s for s in stati if diagnosi(s, soglie)]
    return 1 if critici else 0          # 1 se c'è almeno un host critico, altrimenti 0


# In uno script reale, l'unica riga "a livello di modulo":
#
#     if __name__ == "__main__":
#         sys.exit(cli())
#
# Simulazione della riga di comando dentro al notebook:
codice_uscita = cli(["--cartella", "logs_reali", "--error-rate-max", "0.05"])
print(f"\nExit code: {codice_uscita}  (0 = tutto OK, 1 = almeno un host critico)")


WARNING web02.log:201 riga non valida, saltata: '2025-06-20T15:01:10 host=web02 status=500'
WARNING web02.log:202 riga non valida, saltata: '2025-06-20T15:01:11 host=web02 status=200 rt_ms=NaN'
WARNING web02.log:203 riga non valida, saltata: "QUESTA RIGA E' ROTTA"
WARNING web02.log:204 riga non valida, saltata: '2025-06-20T15:01:13 status=200 rt_ms=55'
INFO Righe scartate in totale: 4
INFO Eventi validi letti: 600


=== Report host ===
web01    | rich:  200 | err:   2.5% | p95:    88ms | OK
web02    | rich:  200 | err:  12.0% | p95:  1538ms | CRITICO
         -> error_rate 12.0% > 5.0%
         -> p95 1538ms > 800ms
web03    | rich:  200 | err:   3.5% | p95:   109ms | OK

Exit code: 1  (0 = tutto OK, 1 = almeno un host critico)


### Un primo assaggio di test

Ecco il pagamento del lavoro fatto: avendo isolato la logica in **funzioni pure**, testarla è
immediato — input noto, output atteso, nessun file o sistema da predisporre. È *impossibile*
farlo sulla versione as-is, dove il calcolo è inchiodato all'I/O.

In un progetto vero questi test stanno in un file `test_monitor.py` e si lanciano con:

```bash
pytest -v
```

`pytest` raccoglie da solo tutte le funzioni `test_*` e segnala quali passano. Qui — per restare
autosufficienti — definiamo gli stessi test e li eseguiamo con un mini-runner inline, ma la
**forma** è esattamente quella di pytest.

Un esempio di minitest automatico:

```python
    def test_parse_riga_valida():
        e = parse_riga("2025-06-20T14:00:01 host=web01 status=200 rt_ms=42")
        assert e == Evento(host="web01", status=200, rt_ms=42)
```

Questo è un **test automatico**: una funzione che verifica, da sola, che un pezzo di codice si comporti come ci aspettiamo. Qui controlliamo che `parse_riga` interpreti correttamente una riga di log ben formata. Smontiamolo.

**Il nome — `test_parse_riga_valida`.** Non è casuale: i framework di test (come `pytest`) cercano automaticamente le funzioni che iniziano con `test_` e le eseguono. Il resto del nome descrive *cosa* stiamo verificando: il parsing di una riga *valida*. Un buon nome di test è una frase che dice quale caso copre.

**La struttura interna segue lo schema classico "Arrange–Act–Assert"**, che vale la pena far notare in aula perché è il telaio di quasi ogni test:

1. **Arrange** - nel test che stiamo guardando, la fase Arrange in pratica non c'è — o meglio, è ridotta al minimo e si confonde con l'Act. È proprio il motivo per cui prima la mettevamo tra parentesi.<br>
Ricapitoliamo cosa sarebbe l'Arrange: è la fase di preparazione, dove allestiamo tutto ciò che serve prima di eseguire la cosa da testare — costruire oggetti di partenza, preparare dati di input, predisporre un file finto, impostare uno stato iniziale. È il "apparecchiare la tavola" prima di agire.

2. **Act** — eseguiamo la cosa da testare:
   ```python
   e = parse_riga("2025-06-20T14:00:01 host=web01 status=200 rt_ms=42")
   ```
   Diamo a `parse_riga` un input **noto e controllato** (una riga perfetta) e ne raccogliamo il risultato in `e`.

3. **Assert** — dichiariamo cosa *deve* essere vero:
   ```python
   assert e == Evento(host="web01", status=200, rt_ms=42)
   ```
   Affermiamo che il risultato sia *esattamente* l'`Evento` atteso.

(La fase "Arrange", cioè la preparazione dei dati, qui è minima — l'input è la stringa stessa — quindi si fonde con l'Act.)

**Il cuore è `assert`.** È un'istruzione di Python che dice: "*questa condizione deve essere vera; se non lo è, fermati e segnala un errore*". Se l'espressione è vera, non succede nulla e il test passa in silenzio. Se è falsa, `assert` solleva un `AssertionError` e il test **fallisce**, segnalando che qualcosa non torna. È il meccanismo con cui il test "boccia" il codice rotto.

**Perché il confronto `e == Evento(...)` funziona** — ed è il punto che lega questo test a tutto il discorso di prima. Stiamo confrontando due oggetti `Evento` con `==`, e ci aspettiamo che risultino uguali se hanno gli stessi campi. Questo è possibile **proprio grazie a `@dataclass`**, che ci ha generato il metodo `__eq__`: senza, `==` confronterebbe l'identità in memoria (due oggetti diversi → sempre diversi) e il test fallirebbe anche con un parsing corretto. Il decoratore di cui parlavamo è ciò che rende questo test scrivibile in modo così naturale.

**Cosa significa, in pratica, se questo test passa:** abbiamo la prova — automatica e ripetibile — che `parse_riga`, davanti a una riga valida, estrae correttamente `host`, `status` e `rt_ms` e li impacchetta nell'`Evento` giusto. E qui si chiude il cerchio con il filo conduttore del modulo: possiamo testare `parse_riga` così facilmente **perché è una funzione pura** — riceve una stringa, restituisce un `Evento`, senza leggere file né dipendere da nient'altro. Darle un input e confrontarne l'output è immediato. Sullo script as-is, dove il parsing era annegato dentro il ciclo di lettura dei file, un test così non sarebbe stato possibile.

Vale la pena aggiungere che questo è **un solo caso** — la riga valida. La robustezza vera nasce affiancandogli i test dei casi-limite (riga con campo mancante, valore non numerico, riga spazzatura), che verificano che `parse_riga` restituisca `None` invece di esplodere: sono esattamente le tre righe sporche che avevamo visto far crashare lo script as-is.

In [ ]:
def test_parse_riga_valida():
    e = parse_riga("2025-06-20T14:00:01 host=web01 status=200 rt_ms=42")
    assert e == Evento(host="web01", status=200, rt_ms=42)

def test_parse_riga_non_numerica():
    assert parse_riga("2025-06-20T14:00:01 host=web01 status=200 rt_ms=NaN") is None

def test_parse_riga_campo_mancante():
    assert parse_riga("2025-06-20T14:00:01 host=web01 status=200") is None      # manca rt_ms

def test_parse_riga_spazzatura():
    assert parse_riga("QUESTA RIGA E' ROTTA") is None

def test_percentile_nearest_rank():
    assert percentile([10, 20, 30, 40, 50], 0.95) == 50
    assert percentile([42], 0.95) == 42

def test_aggrega_kpi():
    eventi = [
        Evento("a", 200, 10),
        Evento("a", 500, 20),   # 1 errore su 4
        Evento("a", 200, 30),
        Evento("a", 200, 40),
    ]
    [stato] = aggrega(eventi, Soglie())
    assert stato.n_richieste == 4
    assert stato.n_errori == 1
    assert stato.error_rate == 0.25


def esegui_test(namespace) -> None:
    """Mini-runner: esegue tutte le funzioni test_* del namespace (al posto di pytest)."""
    falliti = 0
    for nome, fn in sorted(namespace.items()):
        if nome.startswith("test_") and callable(fn):
            try:
                fn()
                print(f"PASS  {nome}")
            except AssertionError as e:
                falliti += 1
                print(f"FAIL  {nome}: {e}")
    print("\n" + ("TUTTI I TEST PASSANO" if not falliti else f"{falliti} TEST FALLITI"))


esegui_test(dict(globals()))


PASS  test_aggrega_kpi
PASS  test_parse_riga_campo_mancante
PASS  test_parse_riga_non_numerica
PASS  test_parse_riga_spazzatura
PASS  test_parse_riga_valida
PASS  test_percentile_nearest_rank

TUTTI I TEST PASSANO


## Confronto: as-is → strumento

| Aspetto | **AS-IS** (script lineare) | **Strumento** |
|---|---|---|
| Organizzazione | un unico blocco | funzioni a responsabilità singola |
| Percorsi e soglie | cablati nel codice (numeri magici) | parametri + `Soglie` (un posto solo) |
| Modello dati | dizionari paralleli da allineare | `dataclass` (`Evento`, `StatoHost`) |
| Parsing | mescolato, assume input perfetto | funzione **pura** e difensiva |
| Lettura file | `open` senza `with`/encoding, legge tutto | `with` + encoding + `glob` + streaming |
| Calcolo | accoppiato all'I/O | funzioni **pure** (`aggrega`, `percentile`) |
| Righe sporche | lo script si interrompe | contate, segnalate, saltate |
| Diagnostica | `print` mescolato all'output | `logging` separato per livello |
| Integrazione | nessun esito programmatico | **exit code** per cron/scheduler |
| Test | impossibile | banale sulle funzioni pure |

## Cosa abbiamo guadagnato (e quando fermarsi)

Non abbiamo aggiunto funzionalità: l'output è lo stesso. Abbiamo cambiato **come** è fatto, e
con questo abbiamo ottenuto uno strumento che **sopravvive ai dati reali**, **dice cosa è andato
storto**, si **integra** con uno scheduler, si **modifica senza paura** (i test lo proteggono) e
si **riusa** come modulo.

Questo **non** significa che ogni script vada ristrutturato così: per un'estrazione *una tantum*
da lanciare a mano, la forma lineare va benissimo — la struttura è un investimento, e ha senso
quando il codice viene **rieseguito**, **cresce**, va **condiviso** o **testato**. Il punto del
modulo è saper riconoscere *quando* fare il salto e *come*.

Lo scheletro costruito qui — **leggi (I/O) → trasforma (puro) → aggrega (puro) → riporta (I/O)**,
con configurazione, logging, gestione errori e test — è la **spina dorsale** di tutto il corso.
Nei prossimi giorni le sorgenti diventeranno Oracle, SQL Server e Splunk, e la lettura verrà
**parallelizzata** su centinaia di host: ma la struttura resterà questa.

# Il flowchart della pipeline idiomatica

In [ ]:
Image('mappa_idioma.png') if IN_COLAB else display(Image(filename='vettorizzazione.png'))

È un **flowchart della pipeline idiomatica**: la spina dorsale `leggi → aggrega → diagnosi → riporta`, con i colori a distinguere ciò che tocca il mondo esterno (I/O ai bordi) da ciò che è logica pura, e due funzioni di supporto agganciate ai lati (`parse_riga` e la configurazione `Soglie`).Come leggerla, e come si lega al notebook:

La **colonna centrale è il flusso dei dati**, dall'alto in basso, ed è esattamente ciò che esegue `main()`: `cli/parse_args` raccoglie gli argomenti → `leggi_eventi` legge i file → `aggrega` calcola i KPI → `diagnosi` decide chi è critico → `stampa_report` produce l'output. Le **frecce portano i tipi** che passano da uno stadio all'altro (`list[Evento]` → `list[StatoHost]`), così si vede la trasformazione progressiva dei dati.

Il **colore codifica il principio cardine del modulo**: coral = funzioni che toccano il mondo esterno (l'I/O ai bordi: lettura file, stampa, exit code), teal = funzioni pure (la logica testabile, al centro). I **due riquadri laterali** sono i supporti: `parse_riga` (pura) che `leggi_eventi` chiama riga per riga, e `Soglie` (la configurazione) che alimenta `aggrega` e `diagnosi`.

Per completezza, i passaggi logici che *non* stanno nella spina ma fanno parte dello strumento:

- **`main()`** — è la colonna stessa: l'orchestratore che mette in fila i quattro stadi senza contenere logica di dettaglio.
- **`cli()` + `if __name__ == "__main__"`** — il guscio che traduce gli argomenti, chiama `main` e restituisce l'**exit code** (0/1) per cron/scheduler.
- **`logging`** — trasversale: `leggi_eventi` lo usa per segnalare le righe scartate, separando la diagnostica dall'output.
- **`percentile`** — la funzione pura di supporto a `aggrega` per la p95.
- **Test (`pytest`)** — verificano le funzioni **pure** (`parse_riga`, `percentile`, `aggrega`): è proprio il colore teal che le rende testabili in isolamento.

Se ti è utile per l'aula, posso esportartela come file **SVG o PNG** da incollare nel notebook o in una slide, oppure prepararne una seconda versione "a fasi" che evidenzi una alla volta le sezioni 2.1 → 2.7 mentre le spieghi.